
# <font color="red"> Redes Neuronales Recurrentes</font>

## <font color="blue"> Por Alfredo Alfredo Diaz </font>

El objetivo de esta sección es aprender y practicar redes neuronales recurrentes (RNN) en diversas tareas y aplicaciones, siendo la primera de ellas el análisis de datos de series temporales. Las RNN han revolucionado la forma en que se pronostica la información secuencial.

* Ver este enlace animado https://joshvarty.github.io/VisualizingRNNs/

* Ver explicación detallada https://colab.research.google.com/drive/1beXNFH-VT7fbefc2_hG8P1VLOTljh3UT?usp=sharing


## 📋 Tabla de Contenidos — Cuaderno 10

> **Posición en el curso**: Este cuaderno es el **Módulo 10** y debe estudiarse **antes** del Módulo 11 (NLP con Embeddings).  
> Aquí aprenderá la base matemática y práctica de las RNNs. El Módulo 11 usará estas redes para procesar lenguaje natural.

---

1. [¿Qué es una RNN? Intuición y arquitectura](#1-que-es-una-rnn)
2. [Matemática de las RNNs: Forward y Backward Pass](#2-matematica)
3. [Tipos de RNN (one-to-one, one-to-many…)](#3-tipos)
4. [Representación de texto: One-Hot vs Embedding](#4-representacion)
5. [Ejemplo 1 — RNN simple: predicción de series temporales](#5-series)
6. [Ejemplo 2 — Generación de nombres con SimpleRNN / LSTM / GRU](#6-nombres)
7. [El Problema del Gradiente Desvanecido](#7-gradiente)
8. [LSTM: arquitectura detallada y ecuaciones](#8-lstm)
9. [GRU: la alternativa eficiente al LSTM](#9-gru)
10. [Ejemplo 3 — Generación de texto (Alicia en el País de las Maravillas)](#10-texto)
11. [Ejemplo 4 — Predicción de precio de acciones (Tesla)](#11-acciones)
12. [¿Cuántas capas y neuronas usar? Guía práctica](#12-guia)
13. [Resumen comparativo y cuándo usar cada arquitectura](#13-resumen)
14. [Ejercicios propuestos](#14-ejercicios)

---
> **Prerrequisitos**: Python, NumPy, TensorFlow/Keras, scikit-learn, matplotlib


---

# 🧠 1. ¿Qué es una Red Neuronal Recurrente? — Intuición

---

## Limitación de las redes feedforward tradicionales

Las redes neuronales **densas (MLP)** y **convolucionales (CNN)** procesan cada entrada de forma **independiente**: no tienen noción del orden ni del contexto anterior.

Esto es un problema cuando los datos tienen estructura **temporal** o **secuencial**:

| Tipo de dato | ¿Por qué importa el orden? |
|---|---|
| Texto | "El banco del río" ≠ "Fui al banco a pagar" |
| Audio / Habla | Las sílabas forman palabras en secuencia |
| Series temporales | El precio de hoy depende del precio de ayer |
| Video | Los fotogramas tienen continuidad temporal |
| ADN | La secuencia de bases importa |

## La solución: memoria interna

Una **RNN** agrega un **estado oculto** $h_t$ que actúa como memoria: captura lo que ocurrió en los pasos anteriores y lo combina con la entrada actual.

```
Paso 1:   x₁ → [RNN] → h₁
Paso 2:   x₂ → [RNN] + h₁ → h₂
Paso 3:   x₃ → [RNN] + h₂ → h₃   → ŷ
```

La misma celda RNN se aplica en **cada paso de tiempo**, compartiendo sus pesos. Esto le permite generalizar a secuencias de cualquier longitud.

## Analogía pedagógica

Imagina que estás leyendo un libro palabra por palabra. Tu cerebro no procesa cada palabra de forma aislada: recuerda las palabras anteriores para entender el significado de la actual.

Una RNN hace algo parecido: **acumula contexto** a medida que avanza en la secuencia.

## Representación visual

```
        ┌─────┐        ┌─────┐        ┌─────┐
h₀ ─→  │ RNN │─ h₁ →  │ RNN │─ h₂ →  │ RNN │─ h₃ → ŷ
        │     │        │     │        │     │
    x₁ ─┘     └─   x₂ ─┘     └─   x₃ ─┘
```

Los pesos **W, U, V** son los mismos en cada paso (parámetros compartidos).




En una red neuronal recurrente, se guardan (memoria) las activaciones de salida de una o más capas de la red. A menudo, estas activaciones son de las capas ocultas. Luego, la próxima vez que alimentamos un ejemplo de entrada a la red, incluimos las salidas almacenadas previamente como entradas adicionales. Estas entradas adicionales como si estuvieran concatenadas al final de las entradas “normales” de la capa anterior.

 Por ejemplo, si una capa oculta tiene 10 nodos de entrada regulares y 128 nodos ocultos en la capa, entonces tendría en realidad un total de 138 entradas. Por supuesto, la primera vez que intentas calcular la salida de la red, necesitarás rellenar esas 128 entradas adicionales con ceros o algo similar.

![image](https://github.com/adiacla/RNN/blob/main/rnn.png?raw=true)



Las redes neuronales recurrentes (En inglés Recurrent Neural Networks - RNN) son una familia de redes neuronales para procesar datos secuenciales, las cuales se basan en el principio de compartir parámetros a lo largo de diferentes partes del modelo, lo que permite aplicar el modelo a datos con estructuras diferentes y generalizar sobre ellos.

Si por el contrario se tuviera un párametro para cada índice de tiempo, no se podría generalizar a longitudes de secuencias no vistas durante el entrenamiento, o compartir patrones detectados por el modelo a lo largo de diferentes longitudes de secuencias y posiciones en el tiempo; dicha propiedad es particularmente importante cuando una pieza de información específica puede ocurrir en múltiples posiciones dentro de una secuencia.

La red permanece Feed-forward pero mantiene el estado interno a través de nodos de contexto los cuales influencian la capa oculta en entradas subsecuentes. Existen diferentes arquitecturas de RNN, las más conocidas son las Elman y Jordan como se aprecia la siguiente imagen.

![image](https://raw.githubusercontent.com/jdariasl/ML_2020/a4f7f015e29b279db380dd51f1a5e379fd3d4ae4//Images/RNN2.png)

# Ejemplo básico de RNN con SimpleRnn

In [ ]:
# ==========================================
# Introducción a RNN: Predicción de series temporales
# ==========================================

import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Input
from sklearn.preprocessing import MinMaxScaler


## Generación de datos

Creamos una serie temporal senoidal con ruido (algo parecido a lo que ocurre con señales físicas o financieras).
Esto nos da un patrón periódico pero imperfecto → ideal para mostrar cómo la RNN aprende dependencias temporales.

In [ ]:

# ==========================================
# Generación de datos sintéticos
# ==========================================

# Función que genera una serie senoidal con algo de ruido
def generate_time_series(n_samples=1000, n_timesteps=50, freq=0.1):
    time = np.linspace(0, n_timesteps * freq, n_timesteps)
    # Serie base: senoidal + ruido
    series = np.sin(2 * np.pi * time) + 0.1 * np.random.randn(n_timesteps)
    return series

# Generamos una serie
series = generate_time_series()

series

In [ ]:
time=np.round(np.linspace(0, 50 * 0.1,50)*10,0)
time

In [ ]:
len(series),series.shape

In [ ]:
# Exploramos visualmente la serie
plt.figure(figsize=(8,4))
plt.plot(series, label='Serie original')
plt.title("Serie temporal sintética (senoidal con ruido)")
plt.xlabel("Tiempo")
plt.ylabel("Valor")
plt.legend()
plt.show()


## Normalización

Las redes neuronales aprenden mejor cuando los datos están en un rango pequeño, por eso usamos MinMaxScaler → [0, 1].

Aunque el rango sea -1 a 1, hay dos razones prácticas por las que se sigue normalizando

a. Cuando hay ruido o desbalance en la escala

b. Por coherencia con pipelines reales: En problemas reales, los datos casi nunca están perfectamente escalados.

Por eso, en la práctica, se normaliza siempre para que la red reciba entradas en una escala controlada.

In [ ]:
# ==========================================
# Preparación de los datos para la RNN
# ==========================================

# Normalizamos los datos para que la red aprenda mejor
scaler = MinMaxScaler(feature_range=(0, 1))
series_scaled = scaler.fit_transform(series.reshape(-1, 1))


Creación de secuencias

Una RNN no trabaja con un solo número, sino con una secuencia de pasos anteriores.
Ejemplo: si window_size=10, la red verá los últimos 10 valores y tratará de predecir el 11º.

In [ ]:
# Función para crear secuencias (ventanas deslizantes)
def create_sequences(data, window_size=10):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size])
    return np.array(X), np.array(y)

window_size = 10
X, y = create_sequences(series_scaled, window_size)

In [ ]:
X[0],X[1],X[2]

In [ ]:
y

In [ ]:
X.shape

In [ ]:
# Redimensionamos X para que tenga la forma [muestras, pasos temporales, características]
X = X.reshape((X.shape[0], X.shape[1], 1))

print(f"Forma de X: {X.shape}")
print(f"Forma de y: {y.shape}")

In [ ]:
X

entonces:

Tienes 40 ventanas (muestras).

Cada ventana tiene 10 pasos de tiempo.

Cada paso tiene 1 característica.

In [ ]:
# Primera ventana
primera_ventana = X[0]        # Toma la primera muestra -> forma (10, 1)
print(primera_ventana.shape)
print(primera_ventana)

## Modelo RNN

La capa SimpleRNN tiene 32 neuronas y usa activación tanh (adecuada para datos continuos).
Después, una capa Dense(1) genera la predicción del siguiente valor.

In [ ]:
# ==========================================
# Construcción del modelo RNN
# ==========================================

model = Sequential([
    Input(shape=(window_size, 1)),
    SimpleRNN(units=32, activation='tanh'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')
model.summary()


In [ ]:
from tensorflow.keras.utils import plot_model
plot_model(
    model,
    to_file="rnn_model.png",
    show_shapes=True,
    show_layer_names=True,
    show_dtype=False,
    show_layer_activations=True
)


![](https://raw.githubusercontent.com/adiacla/bigdata/refs/heads/master/SimpleRNN.png)

## Entrenamiento y validación

Entrenamos con el 80% de los datos y validamos con el 20%.
La pérdida MSE (error cuadrático medio) muestra si la red aprende el patrón de la serie.

In [ ]:
# ==========================================
# Entrenamiento del modelo
# ==========================================

history = model.fit(X, y, epochs=50, batch_size=16, validation_split=0.2, verbose=1)

# Visualizamos la pérdida (MSE)
plt.figure(figsize=(8,4))
plt.plot(history.history['loss'], label='Entrenamiento')
plt.plot(history.history['val_loss'], label='Validación')
plt.title('Evolución del error (MSE)')
plt.xlabel('Épocas')
plt.ylabel('Error')
plt.legend()
plt.show()


## Predicción

Tomamos la última ventana de la serie y predecimos el siguiente valor.
La gráfica muestra el punto rojo como el valor predicho.

In [ ]:
# ==========================================
# Predicción con la RNN
# ==========================================

# Tomamos los últimos 10 puntos de la serie como entrada
last_window = series_scaled[-window_size:].reshape(1, window_size, 1)
predicted_scaled = model.predict(last_window)

# Desnormalizamos la predicción
predicted = scaler.inverse_transform(predicted_scaled)

print(f"Predicción del siguiente valor: {predicted[0][0]:.4f}")

# Visualizamos el resultado
plt.figure(figsize=(8,4))
plt.plot(series, label='Serie original')
plt.axvline(x=len(series)-window_size, color='gray', linestyle='--', label='Ventana usada')
plt.scatter(len(series), predicted[0][0], color='red', label='Predicción')
plt.title('Predicción del siguiente valor de la serie')
plt.xlabel('Tiempo')
plt.ylabel('Valor')
plt.legend()
plt.show()

In [ ]:
# ==========================================
# Predicción con la RNN
# ==========================================

# Tomamos los últimos 'window_size' puntos de la serie como entrada
last_window = series_scaled[-window_size:].flatten().tolist() # Flatten and convert to list

# Lista para almacenar las predicciones futuras
predicted_scaled_values = []

# Realizamos predicciones para los próximos 3 pasos

num_predictions = 9
for _ in range(num_predictions):
    # Reshape the last_window for prediction
    input_sequence = np.array(last_window[-window_size:]).reshape(1, window_size, 1)

    # Predict the next value
    next_prediction_scaled = model.predict(input_sequence, verbose=0)[0][0]

    # Append the prediction to the list
    predicted_scaled_values.append(next_prediction_scaled)

    # Add the predicted value to the last_window to use for the next prediction
    last_window.append(next_prediction_scaled)


# Desnormalizamos las predicciones
predicted_values = scaler.inverse_transform(np.array(predicted_scaled_values).reshape(-1, 1))

print(f"Predicción de los próximos {num_predictions} valores:")
for i, pred in enumerate(predicted_values):
    print(f"Paso {i+1}: {pred[0]:.4f}")


# Visualizamos el resultado
plt.figure(figsize=(10,6))
plt.plot(series, label='Serie original')

# Plot the original series data points used for the last window
plt.plot(np.arange(len(series) - window_size, len(series)), series[-window_size:], color='orange', label='Ventana usada para predicción inicial')



# Plot the predicted values
# We need to determine the x-axis values for the predictions
last_index = len(series) -1
predicted_indices = np.arange(last_index + 1, last_index + 1 + num_predictions)
plt.plot(predicted_indices, predicted_values.flatten(), color='red', linestyle='--', marker='o', label=f'Predicciones ({num_predictions} pasos)')

plt.title(f'Predicción de los próximos {num_predictions} valores de la serie')
plt.xlabel('Tiempo')
plt.ylabel('Valor')
plt.legend()
plt.grid(True)
plt.show()


#Tipos de RNN
Las redes prealimentadas tienen entrada y salida únicas, mientras que las redes neuronales recurrentes son flexibles, ya que se puede cambiar la longitud de las entradas y las salidas. Esta flexibilidad permite a las RNN generar música, clasificación de sentimiento y traducción automática.

Hay cuatro tipos de RNN basados en diferentes longitudes de entradas y salidas.

**Uno a uno** es una red neuronal sencilla. Se suele utilizar para problemas de machine learning que tienen entrada y salida únicas.

**Uno a muchos** tiene entrada única y varias salidas. Se utiliza  por ejemplo en generar comentarios de foto.

**Muchos a uno** toma una secuencia de varias entradas y predice una salida única. Es popular en la clasificación de sentimiento, donde la entrada es texto y la salida es una categoría.

**Muchos a muchos** toma varias entradas y salidas. La aplicación más común es la traducción automática.

![image](https://raw.githubusercontent.com/jdariasl/ML_2020/a4f7f015e29b279db380dd51f1a5e379fd3d4ae4/Images/RNN-Topol.jpeg)

Retomando la definición, una red neuronal recurrente (RNN) es un tipo de red neuronal especializada en el procesamiento de secuencias de datos
$x(t)=x(1),…,x(τ)$, donde el índice de tiempo
$𝑡$ varía desde 1 hasta $τ$. Para tareas que involucran entradas secuenciales, como el habla y el lenguaje, a menudo es mejor utilizar RNNs.

En un problema de procesamiento de lenguaje natural (NLP), si deseas predecir la siguiente palabra en una oración, es importante conocer las palabras que la preceden. Las RNNs se llaman recurrentes porque realizan la misma tarea para cada elemento de una secuencia, con la salida dependiendo de los cálculos previos. Otra forma de pensar en las RNNs es que tienen una "memoria" que captura información sobre lo que se ha calculado hasta el momento.

![image](https://github.com/adiacla/RNN/blob/main/rnn2.png?raw=true)

Tomado del libro de Chollet, F., & Allaire, J. (2021). Recurrent neural networks

El lado izquierdo del diagrama anterior muestra la notación de una RNN y, en el lado derecho, una RNN desenrollada (o desplegada) en una red completa. Es decir si se despliega la red en detalle, se observa toda  la secuencia. Por ejemplo, si la secuencia que nos interesa es una oración de 3 palabras, la red se fija en una red neuronal de 3 capas, una capa para cada palabra.


# Repaso de las fórmulas básicas usadas

## Entrada

x(t) se toma como entrada a la red en el instante de tiempo t. Por ejemplo,  $𝑥_1$  podría ser un vector one-hot correspondiente a una palabra de la oración.

## Estado oculto

h(t) representa el estado oculto en el tiempo 𝑡 y actúa como la "memoria" de la red.

h(t) se calcula en función de la entrada actual y el estado oculto del paso de tiempo anterior:

$h^{(t)}=f(Ux^{(t)}+Wh^{(t−1)})$

La función 𝑓 se elige como una transformación no lineal, como
*tanh* o *ReLU*.

## Pesos

La RNN tiene conexiones de entrada a estado oculto parametrizadas por una matriz de pesos 𝑈, conexiones recurrentes de estado oculto a estado oculto parametrizadas por una matriz de pesos 𝑊, y conexiones de estado oculto a salida parametrizadas por una matriz de pesos 𝑉. Todos estos pesos $(U,V,W)$ se comparten a lo largo del tiempo.

## Salida

o(t) ilustra la salida de la red. En el diagrama, he añadido una flecha después de $o(t)$, que también suele estar sujeta a no linealidad, especialmente cuando la red contiene más capas hacia abajo.


## Paso hacia adelante o Forward Pass

La figura no especifica la elección de la función de activación para las unidades ocultas. Antes de continuar, hacemos algunas suposiciones:

1. Se asumime que la función de activación tangente hiperbólica se utiliza para la capa oculta;

2. Se asumime que la salida es discreta, como si la RNN se utilizara para predecir palabras o caracteres.

Una forma natural de representar variables discretas es considerar que la salida 𝑜 proporciona las probabilidades logarítmicas no normalizadas de cada valor posible de la variable discreta.

Luego, podemos aplicar la operación softmax como un paso de post-procesamiento para obtener un vector $ŷ$  de probabilidades normalizadas sobre la salida.


El paso hacia adelante de la RNN se puede representar mediante el siguiente conjunto de ecuaciones.

$$a^{(t)}= b+Wh^{(t-1)}+Ux^{(t)}$$
$$h^{(t)}= tanh (a^{(t)})$$
$$\sigma^{(t)}= c+Vh^{(t)}$$
$$\hat{y}^{(t)}= softmax(\sigma^{(t)})$$


Este es un ejemplo de una red recurrente que mapea una secuencia de entrada a una secuencia de salida de la misma longitud. La pérdida total para una secuencia dada de valores 𝑥 emparejada con una secuencia de valores 𝑦 sería implemente la suma de las pérdidas en todos los pasos de tiempo.

Asumimos que las salidas $\sigma^{(t)}$ se utilizan como argumento para la función softmax para obtener el vector $\hat{y}$  de probabilidades sobre la salida. También asumimos que la pérdida 𝐿 es la verosimilitud negativa del verdadero objetivo $𝑦^{(𝑡)}$ dada la entrada hasta el momento.



## Paso hacia atrás o Backward Pass

El cálculo del gradiente implica realizar un paso de propagación hacia adelante moviéndose de izquierda a derecha a través del gráfico mostrado, seguido de un paso de propagación hacia atrás moviéndose de derecha a izquierda. El tiempo de ejecución es 𝑂(𝜏) y no puede reducirse mediante paralelización porque el gráfico de propagación hacia adelante es inherentemente secuencial; cada paso de tiempo puede calcularse solo después del anterior.

Los estados calculados en el paso hacia adelante deben almacenarse hasta que se reutilicen durante el paso hacia atrás, por lo que el costo de memoria también es 𝑂(𝜏).

El algoritmo de retropropagación aplicado al gráfico desenrollado con costo 𝑂(𝜏) se llama retropropagación a través del tiempo (back-propagation through time -BPTT).

Debido a que los parámetros se comparten en todos los pasos de tiempo de la red, el gradiente en cada salida depende no solo de los cálculos del paso de tiempo actual, sino también de los pasos de tiempo anteriores.

## Cálculo de Gradientes

Dada nuestra función de pérdida 𝐿, necesitamos calcular los gradientes para nuestras tres matrices de pesos 𝑈,𝑉,𝑊 y los términos de sesgo 𝑏,𝑐 y actualizarlos con una tasa de aprendizaje 𝛼.

Similar a la retropropagación normal, el gradiente nos da una idea de cómo está cambiando la pérdida con respecto a cada parámetro de peso.

Actualizamos los pesos para minimizar la pérdida con la siguiente ecuación:

$$W←W−\alpha\frac{\partial L}{\partial W}$$

Lo mismo se hace para los otros pesos U,V,b,c.

Ahora calculemos los gradientes mediante BPTT para las ecuaciones de la RNN mencionadas anteriormente. Los nodos de nuestro gráfico computacional incluyen los parámetros 𝑈,𝑉,𝑊,𝑏 y 𝑐, así como la secuencia de nodos indexados por 𝑡 para
$x^{(t)},h(t),o^{(t)}$ y $L^{(t)}$ para cada nodo 𝑛, necesitamos calcular el gradiente $\nabla_𝑛𝐿$ de manera recursiva, basándonos en el gradiente calculado en los nodos que lo siguen en el gráfico.

El gradiente con respecto a la salida $o^{(t)}$ se calcula asumiendo que $o^{(t)}$ se utiliza como argumento para la función softmax para obtener el vector $\hat{𝑦}$  de probabilidades sobre la salida.

También se asumime que la pérdida es la verosimilitud negativa del verdadero objetivo $y^{(t)}$.


## Ejemplo
El siguiente gráfico muestra una implementación para predecir la palabra que le sigue a la oración. Por ejemplo:

The student open their ________

Las opciones del modelo serán books con un probabilidad alta o laptops con una más baja. Ver el grafico.


![image](https://github.com/adiacla/RNN/blob/main/generartexto.png?raw=true)

## Implementación de un RNN usando onehot-encoding o embedding

Vamos a implementar una Red Neuronal Recurrente completa  usando Python. Intentaremos construir un modelo de generación de texto utilizando una RNN. Entrenaremos nuestro modelo para predecir la probabilidad de un carácter dado los caracteres precedentes.

En este proyecto se implementa una red neuronal recurrente simple, diseñada para aprender secuencias de caracteres y generar texto similar al entrenado. A modo de ejemplo, se utiliza una lista de nombres personales, los cuales son procesados y empleados para enseñar al modelo a predecir el siguiente carácter dado uno anterior.

#One-hot-econding y Embedding

Antes de iniciar con dos ejemplos (generación de letras y palabras) y la generación de frases completas, debemos entender como codificar X y y para el entrenamiento

## 1. One-hot Encoding

![imagen](https://cdn-images-1.medium.com/max/1600/1*0kkqYg0mGpyvqvrMam2k2A.png)

**Concepto:**

One-hot encoding es una técnica utilizada para representar datos categóricos en un formato que los modelos pueden entender. En NLP, se utiliza para representar palabras o caracteres como vectores en los cuales solo una posición está activa (con valor 1), y todas las demás están inactivas (con valor 0).

Por ejemplo, si tienes un vocabulario de 5 palabras o caracteres, el one-hot encoding representará cada palabra/ carácter como un vector de tamaño 5, donde solo una posición tiene un 1 indicando la presencia de esa palabra o carácter, y las demás posiciones son 0.


**Uso:**

Es útil para representar palabras o caracteres en modelos de aprendizaje automático que no pueden manejar datos no numéricos.

Sin embargo, se usa menos frecuentemente para vocabularios grandes porque se vuelve muy ineficiente en términos de memoria (ya que se necesita mucho espacio para representar todas las palabras de manera dispersa).


**Técnica o Algoritmo:**

Representación de caracteres o palabras como un vector donde cada índice es 0, excepto el índice correspondiente al elemento presente.

Ejemplo de caracteres con vocabulario de 5 letras (a, b, c, d, e):

a = [1, 0, 0, 0, 0]

b = [0, 1, 0, 0, 0]

c = [0, 0, 1, 0, 0]

d = [0, 0, 0, 1, 0]

e = [0, 0, 0, 0, 1]


Ejemplo de One-hot Encoding para predicción de caracteres:
Frase: "abc"

Vocabulario: ['a', 'b', 'c']

Para representar la frase "abc" con one-hot encoding, asignamos un índice a cada carácter:

a = [1, 0, 0]

b = [0, 1, 0]

c = [0, 0, 1]

Esto sería lo que pasa internamente cuando se alimenta al modelo con caracteres. Sin embargo, a medida que el vocabulario crece (por ejemplo, 5000 caracteres posibles), la representación se vuelve muy dispersa (con la mayoría de los valores siendo ceros).



## 2. Embedding

![image](https://miro.medium.com/v2/resize:fit:720/format:webp/1*9jql32OdNOxS4uQvWmHW7g.png)


**Concepto:**`

Embeddings son representaciones densas y continuas de palabras o caracteres. A diferencia de one-hot encoding, Embedding mapea palabras o caracteres a vectores de números reales (en vez de vectores dispersos de ceros y unos).

Estos vectores son aprendidos durante el entrenamiento y están diseñados para capturar relaciones semánticas entre las palabras. Por ejemplo, palabras que son semánticamente similares tendrán vectores similares.


**Uso:**

En lugar de representar palabras como vectores dispersos (one-hot), los embeddings representan palabras o caracteres como vectores más compactos y significativos.

Son muy útiles para representar grandes vocabularios y capturar significados y relaciones semánticas en el texto.

Por ejemplo, un Embedding de palabras podría colocar palabras como "perro" y "gato" en vectores cercanos porque ambas son animales.


**Técnica o Algoritmo:**

Los embeddings se entrenan utilizando redes neuronales durante el entrenamiento del modelo. Algunos métodos conocidos para aprender embeddings de palabras son:

* *Word2Vec:* Un modelo de redes neuronales que genera embeddings de palabras.

* *GloVe*: Otro enfoque basado en matrices de co-ocurrencia.

* *FastText:* Extiende Word2Vec considerando sub-palabras, útil para vocabularios grandes.


**Ejemplo de Embedding para predicción de palabras:**

Supongamos que tienes la siguiente frase:

Frase: *"el perro corre"*

Vocabulario: ['el', 'perro', 'corre']

Tokenización:

"el" → índice 1

"perro" → índice 2

"corre" → índice 3


**Embeddings: **Usamos embeddings de 2 dimensiones (pueden ser más, pero para fines de ejemplo simplificamos):

"el" → [0.1, 0.2]

"perro" → [0.3, 0.4]

"corre" → [0.5, 0.6]


El modelo aprende estas representaciones densas durante el entrenamiento, y en lugar de usar vectores dispersos, alimentamos los vectores densos que capturan el significado de las palabras.


Cuando alimentamos una red neuronal, no usamos one-hot encoding sino los vectores de embeddings. Por ejemplo, "perro" se representaría como un vector [0.3, 0.4] en lugar de un vector de 3 dimensiones como [0, 1, 0].


# Vamos estudiar un poco más de los embeddings y procesamiento de lengaje natural

Por favor ingrese a estos modulos del curso para comprender un poco más sobre embedding.

1. Embedding con gensim

https://colab.research.google.com/drive/1CxH-954pQE90aZtKmqcdJ-J5g5exV_3c?usp=sharing


2. Embedding con RoBERTa

https://colab.research.google.com/drive/1pPsc80VrZ0yUXvmwZGYUJOlPI7D8KnOd?usp=sharing

# <font color="blue"> Ejemplo 1. RNN con Keras: Generar letras a partir de secuencias de letras de Nombres de personas </font>



## Pasos del proceso

**Carga y limpieza de datos:** Se descarga un archivo de texto desde un repositorio remoto que contiene nombres. Cada nombre se separa por palabras, se convierte a minúsculas y se le eliminan los acentos, con el fin de normalizar la información y evitar errores durante el entrenamiento.

**Construcción del alfabeto:** Se extraen todos los caracteres únicos presentes en los nombres, incluyendo el salto de línea como un carácter especial. Esto permite que el modelo aprenda a generar cadenas de texto completas, simulando nombres separados.

**Codificación one-hot:**  Cada carácter se transforma en un vector one-hot, es decir, un vector binario donde sólo una posición está activada. Esto permite representar los caracteres de manera que la red pueda interpretarlos y aprender patrones.

**Definición del modelo:** Se utiliza Keras para crear una red neuronal con:

Una capa SimpleRNN, que recibe el carácter actual y el estado oculto anterior.

Una capa densa Dense con activación softmax, para predecir el siguiente carácter en la secuencia.

**Entrenamiento manual:** El modelo es entrenado mediante un bucle explícito. A diferencia de redes densas, en este caso se procesa un carácter a la vez, permitiendo un control total del estado oculto que se pasa entre los pasos de la secuencia.

**Generación de texto:** Finalmente, se implementa una función que, a partir de un carácter inicial (o vacío), genera una secuencia de caracteres utilizando el modelo entrenado.

### Descripción del modelo RNN a entrenar

En este ejercicio vamos a entrenar una red neuronal recurrente (RNN) que aprenda a generar nuevos textos a partir de un conjunto de ejemplos de nombres.

Utilizaremos una lista de nombres como base de datos (en este caso, nombres de personas), y la red aprenderá a predecir carácter por carácter. Es decir, a partir de un carácter inicial, intentará generar secuencias coherentes que se asemejen a los nombres reales.


### Objetivo del modelo

El propósito principal es que, dada una letra inicial (por ejemplo, “T”), la red sea capaz de predecir letra a letra cómo podría continuar el nombre, hasta formar uno completo.


### Metodología

Dividiremos el desarrollo en las siguientes etapas:

1. **Carga e inspección de los datos**  
   Leeremos los nombres desde un archivo o texto plano y procesaremos cada carácter para poder alimentar la red.

2. **Preprocesamiento y vectorización**  
   Convertiremos los nombres a secuencias de índices numéricos que representen cada carácter (one-hot encoding).

3. **Construcción del modelo RNN**  
   Implementaremos una red recurrente simple utilizando Keras, con una capa `SimpleRNN` o `LSTM`, seguida de una capa densa que predice el siguiente carácter.

4. **Entrenamiento del modelo**  
   Ajustaremos los pesos del modelo usando el conjunto de nombres como secuencias de entrenamiento.

5. **Generación de nombres**  
   Una vez entrenado, usaremos el modelo para generar nombres nuevos a partir de una letra inicial.

Este es un ejemplo ideal para entender cómo funcionan las redes neuronales recurrentes al trabajar con secuencias de texto, y puede adaptarse fácilmente para otros casos como generación de texto, nombres de personas, productos, lugares, etc.

In [ ]:
import os
import random
import unicodedata
import requests
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Input


In [ ]:
#Controlar la reproducibilidad
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# Generador de letras donde entrenamos con un corpus de Nombres con RNN Simple
Este cuaderno muestra cómo construir un modelo de red neuronal recurrente (RNN) simple utilizando `SimpleRNN` de Keras para generar nuevos nombres a partir de una base de datos de nombres en español. Se siguen estos pasos principales:

1. Cargar y limpiar datos desde un archivo remoto.
2. Preprocesar los datos y convertirlos en formato adecuado.
3. Construir y entrenar un modelo `SimpleRNN`.
4. Generar texto (nombres) a partir de semillas iniciales usando el modelo entrenado.


In [ ]:
import re
import unicodedata

# --- Cargar y limpiar nombres ---
url = "https://raw.githubusercontent.com/adiacla/bigdata/refs/heads/master/nombres.txt"
texto = requests.get(url).text.strip().lower()


def limpiar_texto(texto):
    texto = unicodedata.normalize('NFKD', texto)
    texto = ''.join(c for c in texto if not unicodedata.combining(c)).lower()
    texto = texto.replace('«', '').replace('»', '')
    texto = re.sub(r'[^a-zñ\s]', '', texto)  # deja solo letras y espacios
    texto = re.sub(r'\s+', ' ', texto)       # colapsa múltiples espacios
    return texto.strip()                     # quita espacios al inicio y final


nombres = [limpiar_texto(linea.strip()) for linea in texto.splitlines() if linea.strip()]
print("Ejemplos de nombres limpios:", nombres[:10])


**np.random.seed(5):** Fijamos la semilla del generador aleatorio para obtener resultados consistentes en cada ejecución.

**Keras – SimpleRNN:** Esta es la celda recurrente básica que permite capturar información secuencial.

**Keras – Input, Dense, Model:** Para definir la estructura del modelo (entrada, capas y arquitectura).

**Keras – SGD:** Optimizador del gradiente descendente simple, ideal para experimentar.

**to_categorical y backend:** Para convertir etiquetas a formato one-hot y gestionar operaciones internas en Keras.



In [ ]:
# --- Crear corpus usando '\n' como símbolo de fin ---
end_token = '\n'
texto = end_token.join(nombres) + end_token
texto = re.sub(r'\n+', '\n', texto)

In [ ]:
print("Fragmento del corpus:\n", texto[:100])
print("Número total de caracteres:", len(texto))
print("Número de nombres:", texto.count('\n'))

In [ ]:
chars = sorted(list(set(texto)))
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for c, i in char2idx.items()}
print("Caracteres únicos:", chars)


In [ ]:
char2idx

In [ ]:
idx2char

In [ ]:
# --- Crear secuencias ---
SEQLEN = 15
STEP = 1
sentences = []
next_chars = []

for i in range(0, len(texto) - SEQLEN, STEP):
    sentences.append(texto[i: i + SEQLEN])
    next_chars.append(texto[i + SEQLEN])


In [ ]:
sentences

In [ ]:

# --- Vectorización ---
X = np.zeros((len(sentences), SEQLEN, len(chars)), dtype=np.float32)
y = np.zeros((len(sentences), len(chars)), dtype=np.float32)

for i, sentence in enumerate(sentences):
    X[i, np.arange(SEQLEN), [char2idx[c] for c in sentence]] = 1
    y[i, char2idx[next_chars[i]]] = 1


In [ ]:
X

In [ ]:
chars

In [ ]:
from tensorflow.keras.layers import SimpleRNN, Dense, Input
from tensorflow.keras.models import Sequential

model = Sequential([
    Input(shape=(SEQLEN, len(chars))),
    SimpleRNN(128, return_sequences=True),
    SimpleRNN(64),
    Dense(len(chars), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau

callbacks = [
    ModelCheckpoint("model_simpleRNN.h5", save_best_only=True),
    ReduceLROnPlateau(monitor="loss", factor=0.5, patience=3, verbose=1)
]

history = model.fit(
    X, y,
    batch_size=128,
    epochs=80,
    callbacks=callbacks
)


In [ ]:
def sample(preds, temperature):
    preds = np.asarray(preds).astype('float64')
    preds = np.log(preds + 1e-8) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    return np.random.choice(len(preds), p=preds)

def generar_nombre(model, temperature, seed=None, max_length=30):
    if seed is None:
      while True:
          seed = random.choice(nombres).ljust(SEQLEN)[:SEQLEN]
          if not seed.startswith('\n'):
              break
    nombre = seed
    while True:
        x_pred = np.zeros((1, SEQLEN, len(chars)))
        for t, char in enumerate(seed):
            if char in char2idx:
                x_pred[0, t, char2idx[char]] = 1
        preds = model.predict(x_pred, verbose=0)[0]
        next_index = sample(preds, temperature)
        next_char = idx2char[next_index]
        if next_char == end_token or len(nombre) > max_length:
            break
        nombre += next_char
        seed = seed[1:] + next_char
    return nombre.replace('\n', '').strip()


In [ ]:
# --- Generación de nombres ---
print("\n--- Nombres generados ---")
for _ in range(10):
    print(generar_nombre(model, 0.9))

### Generación de nombres

Vamos a crear una función gen que utilice tu modelo RNN entrenado para generar nombres, usando la capa recurrente (RNN) y la capa de salida (Dense + softmax). Como ya tienes el modelo entrenado y cargado, y el vocabulario listo (char2idx, idx2char), vamos a estructurarlo bien.



**Temperatura baja (por ejemplo, temperature = 0.5)**

Predicciones más seguras y predecibles.

El modelo tiende a elegir los caracteres con mayor probabilidad (por ejemplo, si “a” tiene 0.7 y “z” tiene 0.01, elegirá “a” casi siempre).

Los nombres generados suelen ser más coherentes y parecidos a los reales, pero también más repetitivos.

 Ejemplo:

*“maria”, “juan”, “pedro”, “camila”*

(poco riesgo, nombres muy comunes)

**Temperatura alta (por ejemplo, temperature = 1 o mayor)**

*Predicciones más aleatorias y creativas.*

El modelo se atreve a tomar letras con menor probabilidad.

Los nombres pueden ser más variados, pero también más erráticos o inventados.

Ejemplo:

*“marionel”, “juandré”, “pedrilo”, “camirza”*


# El Problema del Gradiente Desvanecido

El desvanecimiento del gradiente ocurre cuando los gradientes (derivadas parciales de la función de pérdida con respecto a los parámetros del modelo) se vuelven extremadamente pequeños durante el proceso de retropropagación. Esto puede hacer que las actualizaciones de los pesos sean muy pequeñas, lo que ralentiza o incluso detiene el aprendizaje.

En una red neuronal recurrente como SimpleRNN, el problema se amplifica debido a las conexiones recurrentes, donde la información se "pasa" de un paso de tiempo a otro. Los gradientes que se calculan en cada paso hacia atrás pueden disminuir exponencialmente a medida que se retropropagan a través de muchos pasos de tiempo.

## Efectos del desvanecimiento del gradiente en un modelo SimpleRNN:

**Dificultad para aprender dependencias a largo plazo:**

* El SimpleRNN tiene una estructura que hace que los gradientes se desvanecen rápidamente, especialmente si hay muchas capas o pasos de tiempo en la secuencia.

* Esto impide que el modelo aprenda patrones o dependencias que ocurren a largo plazo (por ejemplo, relaciones entre palabras o eventos distantes en el tiempo).

**Entrenamiento lento o estancado:**

* Debido al desvanecimiento del gradiente, el modelo puede experimentar lentitud en el entrenamiento, ya que las actualizaciones de los pesos se hacen muy pequeñas y los pesos no cambian de manera significativa.

* Esto puede llevar a que el modelo se quede atrapado en un mínimo local o simplemente no converja correctamente durante el entrenamiento.

**Desempeño deteriorado:**

* Los modelos con desvanecimiento de gradiente tienden a no generalizar bien, ya que no pueden capturar patrones complejos o dependencias a largo plazo.

* El modelo tiende a predecir de manera mediocre o incluso de manera completamente incorrecta, especialmente en tareas de predicción de secuencias largas o complejas.


<img src="https://github.com/adiacla/RNN/blob/main/desv_gradiente.jpg?raw=true">


## Soluciones al desvanecimiento del gradiente:
Para mitigar el problema del desvanecimiento del gradiente en redes neuronales recurrentes, hay varias estrategias y soluciones que se pueden aplicar:

**Usar LSTM o GRU en lugar de SimpleRNN:**

LSTM (Long Short-Term Memory) y GRU (Gated Recurrent Unit) son variantes de las redes neuronales recurrentes diseñadas específicamente para manejar el problema del desvanecimiento del gradiente. Estas arquitecturas tienen puertas que permiten que las redes recuerden información durante períodos de tiempo más largos sin que los gradientes se desvanezcan.

Las LSTM y GRU son más eficientes para capturar dependencias a largo plazo en secuencias de datos, ya que tienen mecanismos internos que controlan cómo se mantiene y actualiza la memoria a lo largo del tiempo.

**Inicialización adecuada de los pesos:**

Una buena inicialización de los pesos (por ejemplo, usando He initialization o Xavier initialization) puede ayudar a reducir el riesgo de desvanecimiento del gradiente, al asegurar que los gradientes no se vuelvan demasiado pequeños desde el principio.

**Uso de funciones de activación diferentes:**

La función sigmoide y la tangente hiperbólica (tanh) pueden contribuir al desvanecimiento del gradiente. Usar activaciones como ReLU (Rectified Linear Unit) en las capas recurrentes puede ayudar a mitigar este problema, ya que no tienen saturación en la parte positiva, lo que mantiene los gradientes más grandes y estables.

**Técnicas de normalización:**

La normalización de lotes (Batch Normalization) y la normalización de gradientes pueden ayudar a evitar que los gradientes se desvanezcan, aunque estas técnicas son más comunes en redes neuronales tradicionales, no siempre son aplicables directamente a las RNN debido a la naturaleza secuencial de los datos.

**Recorte de gradientes (Gradient Clipping):**

El recorte de gradientes es una técnica en la que se limita el valor de los gradientes si exceden un cierto umbral. Esto no elimina el desvanecimiento del gradiente, pero ayuda a evitar problemas de explosión del gradiente, otro problema relacionado que puede ocurrir en redes profundas.

## <font color="blue">Memoria a Largo y Corto Plazo (Long/Short Term Memory LSTM) </font>

##Arquitecturas Básica  RNN

Los módulos de repetición RNN sencillos tienen una estructura básica con una capa tanh única. La estructura sencilla de la RNN adolece de memoria corta, por lo que le cuesta retener la información de pasos temporales anteriores en datos secuenciales más numerosos.


![](https://miro.medium.com/v2/resize:fit:1170/1*tvIjQjHW-1WGECloYD5efQ.png)

![](https://media.springernature.com/lw685/springer-static/image/art%3A10.1007%2Fs44196-023-00374-8/MediaObjects/44196_2023_374_Fig3_HTML.png?as=webp)

La Long short-term memory (LSTM) es el tipo avanzado de RNN, que se diseñó para evitar tanto los problemas de desvanecimiento como los de explosión de gradiente. Al igual que la RNN, la LSTM tiene módulos que se repiten, pero la estructura es diferente.

En lugar de tener una capa única de tanh, la LSTM tiene cuatro capas que interactúan y se comunican. Esta estructura de cuatro capas ayuda a la LSTM a conservar la memoria a largo plazo y puede utilizarse en varios problemas secuenciales, como la traducción automática, la síntesis del habla, el reconocimiento del habla y el reconocimiento de la escritura a mano. Puedes adquirir experiencia práctica en LSTM siguiendo la guía: LSTM Python para predicciones bursátiles.

Cada una de las tres puertas (entrada, salida y olvido) puede entenderse como una "neurona" artificial convencional, como en una red neuronal multicapa o de propagación hacia adelante (feedforward): es decir, calculan una activación (mediante una función de activación) de una suma ponderada. Intuitivamente, pueden considerarse como reguladores del flujo de valores que atraviesan las conexiones de la LSTM; de ahí la denominación de "puerta". Existen conexiones entre estas puertas y la celda.

La expresión "memoria a largo y corto plazo" se refiere a que LSTM es un modelo para la memoria de corto plazo que puede durar un largo periodo de tiempo. Las LSTM son ideales para clasificar, procesar y predecir series temporales cuando existen desfases temporales de tamaño y duración desconocidos entre eventos importantes. Las LSTM fueron desarrolladas para lidiar con el problema del gradiente desvanecido y el problema del gradiente explosivo al entrenar redes neuronales recurrentes tradicionales.


![adiaz](https://github.com/adiacla/bigdata/blob/master/lstm.JPG?raw=true)



Referencias:

Deep NLP: Redes LSTM (Memoria a Largo y Corto Plazo) con Matemáticas - Medium
[link text](https://https://medium.com/deep-math-machine-learning-ai/chapter-10-1-deepnlp-lstm-long-short-term-memory-networks-with-math-21477f8e4235)



##Componentes de una celda LSTM

La celda LSTM contiene los siguientes componentes:

![imagen](https://substackcdn.com/image/fetch/$s_!c-Qp!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fbucketeer-e05bbc84-baa3-437e-9518-adb32be77984.s3.amazonaws.com%2Fpublic%2Fimages%2F1f18455e-0e2e-4c0b-86d4-957f4d5d3d93_1122x497.png)

**Puerta de olvido (f):**
Una red neuronal con activación sigmoide que decide qué parte de la información del estado de memoria anterior debe ser descartada.

**Puerta de entrada (i):**
Una red neuronal con activación sigmoide que determina cuánta información nueva se incorporará al estado de memoria.

**Capa candidata (C̃):**
Una red neuronal con activación tanh que genera nuevos valores candidatos para actualizar el estado de memoria.

**Actualización del estado de memoria (C):**
Se calcula combinando la información retenida (tras pasar por la puerta de olvido) con los nuevos valores candidatos ponderados por la puerta de entrada.

**Puerta de salida (o):**
Una red neuronal con activación sigmoide que controla qué parte del estado de memoria se usará para calcular el estado oculto.

**Estado oculto (h):**
El vector de salida en el paso de tiempo actual, calculado a partir del estado de memoria actualizado y filtrado por la puerta de salida.


###**Entradas de la celda LSTM en cada paso:**

𝑋𝑡: entrada actual

𝐻𝑡−1: estado oculto anterior


𝐶𝑡−1: estado de memoria anterior
Salidas de la celda LSTM:

𝐻𝑡: estado oculto actual

𝐶𝑡: estado de memoria actual

Estos componentes trabajan juntos para permitir que la LSTM retenga información relevante a lo largo de pasos de tiempo y olvide aquella que no es útil, facilitando el manejo de dependencias a largo plazo en las secuencias de datos.

## Funcionamiento de las Puertas en una Celda LSTM

**Puerta de Olvido (f):**
Primero, la celda LSTM toma el estado de memoria anterior
𝐶𝑡−1 y lo multiplica, elemento a elemento, por el valor de la puerta de olvido (f) para decidir el estado de memoria actual
𝐶𝑡. La puerta de olvido tiene valores entre 0 y 1:
Si el valor de la puerta de olvido es 0, el estado de memoria anterior se olvida completamente.
Si el valor de la puerta de olvido es 1, el estado de memoria anterior se conserva por completo en la celda.
Esto permite que la celda decida cuánta información del pasado debe mantener o descartar.




**C<sub>t</sub> = C<sub>t-1</sub> * f<sub>t</sub>**

Cálculo del Nuevo Estado de Memoria:

**C<sub>t</sub> = C<sub>t</sub> + (I<sub>t</sub> * C\`<sub>t</sub>)**


Ahora, calculamos la salida:

**H<sub>t</sub> = tanh(C<sub>t</sub>)**




# Ejemplo generacion de nombre con LSTM

In [ ]:
# --- Modelo LSTM ---
model = Sequential([
    Input(shape=(SEQLEN, len(chars))),
    LSTM(128, return_sequences=True),
    LSTM(64),
    Dense(len(chars), activation='softmax')
])


model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


# Visualizamos la arquitectura del modelo
model.summary()

In [ ]:
# --- Entrenar modelo ---
history = model.fit(X, y, batch_size=128, epochs=50)


In [ ]:
# --- Funciones de muestreo y generación ---
def sample(preds, temperature):
    preds = np.asarray(preds).astype('float64')
    preds = np.log(preds + 1e-8) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    return np.random.choice(len(preds), p=preds)

def generar_nombre(model, temperature, seed=None, max_length=30):
    if seed is None:
      while True:
          seed = random.choice(nombres).ljust(SEQLEN)[:SEQLEN]
          if not seed.startswith('\n'):
              break
    nombre = seed
    while True:
        x_pred = np.zeros((1, SEQLEN, len(chars)))
        for t, char in enumerate(seed):
            if char in char2idx:
                x_pred[0, t, char2idx[char]] = 1
        preds = model.predict(x_pred, verbose=0)[0]
        next_index = sample(preds, temperature)
        next_char = idx2char[next_index]
        if next_char == end_token or len(nombre) > max_length:
            break
        nombre += next_char
        seed = seed[1:] + next_char
    return nombre.replace('\n', '').strip()


In [ ]:
# --- Generación de nombres ---
print("\n--- Nombres generados ---")
for _ in range(10):
    print(generar_nombre(model, 0.9))

In [ ]:
# --- Generación de nombres ---
print("\n--- Nombres generados ---")
for _ in range(10):
    print(generar_nombre(model, 1))

# Ejemplo con GRU


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Input

# --- Modelo GRU ---
model = Sequential([
    Input(shape=(SEQLEN, len(chars))),
    GRU(128, return_sequences=True),
    GRU(64),
    Dense(len(chars), activation='softmax')
])

# Compilación
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Visualización del modelo
model.summary()

In [ ]:
# --- Entrenar modelo ---
history = model.fit(X, y, batch_size=128, epochs=50)


In [ ]:
# --- Funciones de muestreo y generación ---
def sample(preds, temperature):
    preds = np.asarray(preds).astype('float64')
    preds = np.log(preds + 1e-8) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    return np.random.choice(len(preds), p=preds)

def generar_nombre(model, temperature, seed=None, max_length=30):
    if seed is None:
      while True:
          seed = random.choice(nombres).ljust(SEQLEN)[:SEQLEN]
          if not seed.startswith('\n'):
              break
    nombre = seed
    while True:
        x_pred = np.zeros((1, SEQLEN, len(chars)))
        for t, char in enumerate(seed):
            if char in char2idx:
                x_pred[0, t, char2idx[char]] = 1
        preds = model.predict(x_pred, verbose=0)[0]
        next_index = sample(preds, temperature)
        next_char = idx2char[next_index]
        if next_char == end_token or len(nombre) > max_length:
            break
        nombre += next_char
        seed = seed[1:] + next_char
    return nombre.replace('\n', '').strip()


In [ ]:
# --- Generación de nombres ---
print("\n--- Nombres generados ---")
for _ in range(10):
    print(generar_nombre(model, 0.9))

In [ ]:
# --- Generación de nombres ---
print("\n--- Nombres generados ---")
for _ in range(10):
    print(generar_nombre(model, 1))

# <font color="green"> Ejemplo 2: Construcción de palabras en secuencia usando LSTM</font>

Fuente de texto: Alicia en el país de las maravillas

Objetivo: Entrenar una red neuronal recurrente (RNN) para generar texto similar al estilo del libro original, pero usando usando Simple, LSTM y GRU

## Lectura del Archivo:

Se obtiene el contenido del archivo de texto desde la URL proporcionada. El texto se normaliza eliminando espacios y convirtiendo a minúsculas.


In [ ]:
import requests
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, GRU, LSTM
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [ ]:
import unicodedata

# === 1. Descargar texto ===
url = "https://github.com/adiacla/RNN/raw/refs/heads/main/APMaravillas.txt"
response = requests.get(url)

def limpiar_texto(texto):
    texto = unicodedata.normalize('NFKD', texto)
    texto = ''.join(c for c in texto if not unicodedata.combining(c))
    texto = texto.replace('«', '').replace('»', '')
    return texto.lower().strip()

text = limpiar_texto(response.text)
text = re.sub(r'\n+', '\n', text)
print("Texto cargado con longitud:", len(text))
print(text[0:500])


## Procesamiento del Texto:

Se divide el texto en líneas y se eliminan las líneas vacías. Luego, todas las líneas se combinan en una sola cadena de texto.

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
encoded = tokenizer.texts_to_sequences([text])[0]
vocab_size = len(tokenizer.word_index) + 1
print("Tamaño del vocabulario:", vocab_size)

In [ ]:
for word, index in list(tokenizer.word_index.items())[:20]:
    print(f"{index}: {word}")


In [ ]:
tokenizer.word_index

In [ ]:
print("Número total de tokens:", len(encoded))
print("Ejemplo de tokens:", encoded[:20])


In [ ]:
# --- Crear secuencias ---
SEQLEN = 40
STEP = 1

X = []
y = []

for i in range(0, len(texto) - SEQLEN, STEP):
    seq_in = texto[i:i + SEQLEN]
    seq_out = texto[i + SEQLEN]
    X.append([char2idx[ch] for ch in seq_in])
    y.append(char2idx[seq_out])

X = np.array(X, dtype=np.int32)
y = np.array(y, dtype=np.int32)

print(" Datos listos")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Ejemplo X[0]:", X[0][:10])
print("Ejemplo y[0]:", y[0])


In [ ]:
print(X.shape)  # (num_samples, SEQLEN)
print(y.shape)  # (num_samples,)


In [ ]:
X[1000]

In [ ]:
Y[1000]

## Construcción del Modelo:

Se crea un modelo secuencial con una capa SimpleRNN que toma las entradas y devuelve la probabilidad de los siguientes caracteres. La salida pasa a través de una capa densa con activación softmax.

## Entrenamiento del Modelo:

El modelo se entrena varias veces (definido por NUM_ITERATIONS), lo que permite que aprenda mejor las relaciones en los datos.

En cada iteración, el modelo se ajusta a los datos de entrada X y las etiquetas y durante un número específico de épocas (NUM_EPOCHS_PER_ITERATION).



In [ ]:
# === Hiperparámetros globales ===
# === Hiperparámetros globales ===
HIDDEN_SIZE = 128
EMBEDDING_DIM = 100

vocab_size = len(char2idx)  # asegúrate de que esté definido
print("Tamaño del vocabulario:", vocab_size)

# === Definir modelo ===
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, GRU, LSTM, Dense

def crear_modelo(tipo):
    model = Sequential()
    model.add(Embedding(input_dim=vocab_size, output_dim=EMBEDDING_DIM, input_length=SEQLEN))

    if tipo == 'SimpleRNN':
        model.add(SimpleRNN(HIDDEN_SIZE))
    elif tipo == 'GRU':
        model.add(GRU(HIDDEN_SIZE))
    elif tipo == 'LSTM':
        model.add(LSTM(HIDDEN_SIZE))
    else:
        raise ValueError("Tipo debe ser 'SimpleRNN', 'GRU' o 'LSTM'")

    model.add(Dense(vocab_size, activation='softmax', dtype='float32'))

    model.compile(
        loss='sparse_categorical_crossentropy',  # y = enteros
        optimizer='adam',
        metrics=['accuracy']
    )
    return model


In [ ]:
# === Entrenamiento con callbacks ===
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import time

modelos = ['SimpleRNN', 'GRU', 'LSTM']

modelos_entrenados = {}
histories = {}

for tipo in modelos:
    print(f"\n Entrenando modelo {tipo}...")

    model = crear_modelo(tipo)

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        ModelCheckpoint(f"best_{tipo}.keras", save_best_only=True)
    ]

    history = model.fit(
        X, y,
        epochs=50,
        batch_size=256,
        validation_split=0.1,
        callbacks=callbacks,
        verbose=1
    )

    histories[tipo] = history
    modelos_entrenados[tipo] = model  # ✅ guardamos el modelo



In [ ]:
modelos_entrenados.keys()
# → dict_keys(['SimpleRNN', 'GRU', 'LSTM'])


In [ ]:
# === 8. Graficar entrenamiento ===
import matplotlib.pyplot as plt

# === 1️⃣ Función para graficar ===
def plot_histories(histories):
    plt.figure(figsize=(14, 6))

    # --- Pérdida ---
    plt.subplot(1, 2, 1)
    for tipo, history in histories.items():
        plt.plot(history.history['loss'], label=f'{tipo} Entrenamiento')
        plt.plot(history.history['val_loss'], '--', label=f'{tipo} Validación')
    plt.title('Evolución de la Pérdida (Loss)')
    plt.xlabel('Épocas')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    # --- Precisión ---
    plt.subplot(1, 2, 2)
    for tipo, history in histories.items():
        plt.plot(history.history['accuracy'], label=f'{tipo} Entrenamiento')
        plt.plot(history.history['val_accuracy'], '--', label=f'{tipo} Validación')
    plt.title('Evolución de la Precisión (Accuracy)')
    plt.xlabel('Épocas')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

# === 2 Llamar a la función ===
plot_histories(histories)


In [ ]:
# === 4. Función para muestreo con temperatura ===
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

def sample_with_temperature(preds, temperature=1.0):
    preds = np.asarray(preds).astype("float64")
    preds = np.log(preds + 1e-8) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    return np.random.choice(len(preds), p=preds)


def predecir_texto(model, seed_text, num_words, tokenizer, temperature=1.0):
    result = []
    text = seed_text

    for _ in range(num_words):
        encoded = tokenizer.texts_to_sequences([text])[0]
        encoded = pad_sequences([encoded], maxlen=SEQLEN, truncating='pre')
        preds = model.predict(encoded, verbose=0)[0]
        next_index = sample_with_temperature(preds, temperature)
        next_word = tokenizer.index_word.get(next_index, '')

        text += ' ' + next_word
        result.append(next_word)

    return seed_text + ' ' + ' '.join(result)


#Función de temperatura


```python
TEMPERATURE = 0.5  # Muy predecible, conservador
TEMPERATURE = 1.0  # Equilibrado
TEMPERATURE = 1.5  # Más loco y creativo```



## Generación de texto:

Al final de todas las iteraciones, se imprime una nueva línea para mejorar la legibilidad de la salida.
Este código te permite entrenar un modelo de generación de texto en español utilizando una red neuronal recurrente.

In [ ]:
# === 9. Generar texto ===
seed_text = "cada vez que el narrador intentaba,seca ya la"
seed_text = seed_text[:SEQLEN].lower()

modelo = modelos_entrenados['SimpleRNN']


generated = predecir_texto(modelo, seed_text, num_words=100, tokenizer=tokenizer, temperature=0.8)
print(generated)


Como se observa el modelo tiene inconvenientes en la construcción de frases completas, esto ocurre por el desvanecimiento del gradiente (vanishing gradient) es un fenómeno común en redes neuronales profundas, especialmente en modelos recurrentes como el SimpleRNN. Este problema puede tener efectos negativos en el proceso de entrenamiento, dificultando que el modelo aprenda relaciones a largo plazo en secuencias de datos, como texto o series temporales.

In [ ]:
# === 9. Generar texto ===
seed_text = "cada vez que el narrador intentaba,seca ya la"
seed_text = seed_text[:SEQLEN].lower()

modelo = modelos_entrenados['LSTM']


generated = predecir_texto(modelo, seed_text, num_words=100, tokenizer=tokenizer, temperature=0.8)
print(generated)


In [ ]:
# === 9. Generar texto ===
seed_text = "cada vez que el narrador intentaba,seca ya la"
seed_text = seed_text[:SEQLEN].lower()

modelo = modelos_entrenados['GRU']


generated = predecir_texto(modelo, seed_text, num_words=100, tokenizer=tokenizer, temperature=0.8)
print(generated)


# <font color="green">Ejercicio con Serie de tiempo para precio de las acciones
Modelo de predicción de precios de acciones utilizando una red neuronal recurrente (RNN), específicamente una LSTM (Long Short-Term Memory). }

Este modelo tiene como objetivo predecir el precio de las acciones de Tesla basándose en datos históricos.

 A continuación explico paso a paso qué hace cada parte del código:

In [ ]:
! pip list | grep "yfinance"

#!pip install yfinance

# 1. Importación de bibliotecas:
Se importan varias bibliotecas esenciales para el procesamiento de datos (numpy, pandas), gráficos (matplotlib, plotly), preprocesamiento de datos (MinMaxScaler de sklearn), y para la creación y entrenamiento del modelo (keras).



In [ ]:
# Importando las bibliotecas
import yfinance as yf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from keras.models import Sequential
from keras.layers import Dense, LSTM, Dropout
from sklearn.metrics import mean_squared_error
import math



# 2. Descarga de los datos históricos de Tesla:
Utiliza la API de Yahoo Finance (yfinance) para descargar los precios históricos de las acciones de Tesla desde enero de 2023 hasta la fecha actual (noviembre de 2024).
Los datos descargados incluyen precios de apertura, cierre, máximo, mínimo, volumen de transacciones, etc. Se guarda en un DataFrame de pandas.

(https://finance.yahoo.com/quote/TSLA/)


In [ ]:
import yfinance as yf

# Descargar los datos históricos de las acciones de Tesla
tesla = yf.Ticker('TSLA')

# Obtener los precios históricos para el periodo desde 2023 hasta la fecha actual
dataset = tesla.history(start="2023-01-01")  # Ajustar la fecha de fin a la actual

dataset



In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
plt.plot(dataset['Close'])
plt.title('Precio de cierre de Tesla')
plt.xlabel('Fecha')
plt.ylabel('Precio de cierre')

# Set y-axis limits
plt.ylim(0)
plt.show()


In [ ]:

# También puedes guardar los datos en un archivo CSV si lo deseas
dataset.to_csv('tesla_stock_data.csv')

In [ ]:

dataset.head()

In [ ]:
dataset[:'2025']

# 3. Preprocesamiento de los datos:
Se separan los datos en dos conjuntos:
Conjunto de entrenamiento: Los datos anteriores a 2023.
Conjunto de prueba: Los datos de 2024 en adelante.
Se selecciona la columna de precios "High" (el precio más alto del día) para predecir el comportamiento de los precios.

In [ ]:
# Usar columna 'Close' para precios de cierre
# Usamos la columna 'Close'
prices = dataset['Close'].values.reshape(-1, 1)


In [ ]:
prices.shape

In [ ]:
# Dividir en 80% train y 20% test
split = int(len(prices) * 0.8)
training_set = prices[:split]
test_set = prices[split:]
print(f'Tamaño train: {len(training_set)}')
print(f'Tamaño test: {len(test_set)}')

In [ ]:
# Graficar precios de cierre
plt.figure(figsize=(12,6))
plt.plot(dataset.index[:split], training_set, label='Train')
plt.plot(dataset.index[split:], test_set, label='Test')
plt.title('Precio de cierre de Tesla - Train/Test')
plt.legend()
plt.show()

In [ ]:
import plotly.graph_objects as go

# Crear un gráfico de velas (candlestick chart)
fig = go.Figure(data=[go.Candlestick(
    x=dataset.index,
    open=dataset['Open'],
    high=dataset['High'],
    low=dataset['Low'],
    close=dataset['Close'],
    increasing_line_color='green', decreasing_line_color='red',  # Colores para las velas
)])

# Agregar título y etiquetas
fig.update_layout(
    title='Tesla Stock Price (2023-2025)',
    xaxis_title='Fecha',
    yaxis_title='Precio de la acción (USD)',
    xaxis_rangeslider_visible=False  # Opcional: desactiva el control deslizante en el gráfico
)

# Mostrar el gráfico
fig.show()

Luego, los precios son normalizados (escalados) para que los valores estén entre 0 y 1 utilizando MinMaxScaler. Esto es importante para mejorar la eficiencia y el rendimiento del modelo de redes neuronales.

In [ ]:
# Escalando el conjunto de entrenamiento
scaler = MinMaxScaler(feature_range=(0, 1))
training_set_scaled = scaler.fit_transform(training_set)
test_set_scaled = scaler.transform(test_set)

In [ ]:
print(training_set_scaled.shape)


# 4. Creación de la estructura de datos para el modelo:
La red LSTM necesita una secuencia de entradas para aprender de los patrones temporales. Por ello, se crean secuencias de 60 días de precios pasados (60 pasos de tiempo) y el precio de cierre correspondiente para cada secuencia.
Esto se hace en el bloque de código siguiente, donde se construye la entrada X_train y la salida y_train.
python
Copiar código


In [ ]:
# Dado que los LSTM almacenan el estado de memoria a largo plazo, creamos una estructura de datos con 30 pasos de tiempo y 1 salida
# Así que, para cada elemento del conjunto de entrenamiento, tenemos 60 elementos previos del conjunto de entrenamiento
SEQLEN = 30  # Por ejemplo, usar 30 días para predecir el siguiente

X_train, y_train = [], []
for i in range(SEQLEN, len(training_set_scaled)):
    X_train.append(training_set_scaled[i-SEQLEN:i, 0])
    y_train.append(training_set_scaled[i, 0])

X_train, y_train = np.array(X_train), np.array(y_train)


In [ ]:
X_train[:3]

In [ ]:
y_train[:3]

Finalmente, X_train se reorganiza en una forma adecuada para la entrada de la red neuronal LSTM, que requiere una estructura de tres dimensiones (muestras, pasos de tiempo, características).

In [ ]:
# Suponiendo que X_train tiene forma (n_samples, timesteps)
# Reshape para LSTM: (samples, time_steps, features)
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))

In [ ]:
X_train

# 5. Construcción del modelo LSTM:
Se define un modelo de red neuronal secuencial con capas LSTM. Las redes LSTM son ideales para datos secuenciales como los precios de acciones, ya que pueden recordar información de pasos anteriores.
Se agregan 4 capas LSTM con regularización de Dropout para evitar el sobreajuste, y finalmente, una capa densa para obtener la salida (predicción del precio).

El modelo se compila utilizando el optimizador RMSprop y la función de pérdida

In [ ]:
from tensorflow.keras.layers import Input

model = Sequential([
    Input(shape=(SEQLEN, 1)),  # Aquí defines la forma de entrada explícitamente
    LSTM(units=50, return_sequences=True),
    Dropout(0.2),
    LSTM(units=50),
    Dropout(0.2),
    Dense(1)
])

model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()

# 6. Entrenamiento del modelo:
El modelo se entrena utilizando el método .fit(), lo que ajusta el modelo a los datos de entrenamiento (X_train y y_train).
También se emplea un división de validación durante el entrenamiento (20%) para monitorear el rendimiento del modelo en datos no vistos mientras se entrena.

In [ ]:
# Separar los últimos N días del entrenamiento como validación (por ejemplo, 20%)
history = model.fit(X_train, y_train, epochs=30, batch_size=32, validation_split=0.1)

# 7. Graficar el history del entrenamiento:
Después de entrenar el modelo, se grafican las curvas de pérdida (loss) tanto para los datos de entrenamiento como para los de validación. Esto ayuda a ver si el modelo está aprendiendo de manera efectiva o si está sobreajustando los datos.
python
Copiar código


In [ ]:
# Graficar el history de la pérdida de entrenamiento y validación
plt.figure(figsize=(10,6))
plt.plot(history.history['loss'], label='Pérdida de Entrenamiento')
plt.plot(history.history['val_loss'], label='Pérdida de Validación')
plt.title('Evolución de la Pérdida durante el Entrenamiento')
plt.xlabel('Épocas')
plt.ylabel('Pérdida')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Unir último bloque de train + test para tener las secuencias de entrada completas
dataset_total = np.concatenate((training_set_scaled, test_set_scaled), axis=0)

X_test = []
for i in range(len(training_set_scaled), len(dataset_total)):
    X_test.append(dataset_total[i-SEQLEN:i, 0])

X_test = np.array(X_test)
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))

# Predicciones
predicted_prices = model.predict(X_test)
predicted_prices = scaler.inverse_transform(predicted_prices)


**8. Gráficos adicionales:**
Se incluyen gráficos para visualizar los precios históricos de las acciones de Tesla, tanto para el conjunto de entrenamiento como para el conjunto de prueba, antes y después de 2024.
También, se muestra un gráfico tipo candlestick utilizando Plotly para visualizar los precios abiertos, cerrados, altos y bajos de las acciones de Tesla.

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(test_set, color='red', label='Precio real')
plt.plot(predicted_prices, color='blue', label='Precio predicho')
plt.title('Predicción Precio Acciones Tesla')
plt.xlabel('Tiempo')
plt.ylabel('Precio Tesla')
plt.legend()
plt.show()

# Calcular RMSE
rmse = math.sqrt(mean_squared_error(test_set, predicted_prices))
print(f'RMSE: {rmse}')


In [ ]:
# Suponiendo que test_set es un array 2D con forma (num_muestras, 1)
last_days = dataset['Close'][-SEQLEN:].values  # últimos SEQLEN precios reales

# Normalizamos esos últimos días igual que entrenamiento
last_days_scaled = scaler.transform(last_days.reshape(-1,1))

predictions_scaled = []

input_seq = last_days_scaled.reshape(1, SEQLEN, 1)  # forma (1, SEQLEN, 1)

for _ in range(10):  # predecir 10 días futuros
    pred = model.predict(input_seq)[0,0]
    predictions_scaled.append(pred)

    # actualizar input_seq para siguiente predicción: se quita primer valor, se añade predicción
    input_seq = np.append(input_seq[:,1:,:], [[[pred]]], axis=1)

# Volver a escala original
predictions = scaler.inverse_transform(np.array(predictions_scaled).reshape(-1,1))

print("Predicciones para los próximos 10 días:")
print(predictions.flatten())


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Parámetros
SEQLEN = 30  # longitud de la secuencia que usaste para entrenar el modelo
forecast_days = 30

# Última fecha del dataset
last_date = dataset.index[-1]

# Últimos SEQLEN precios de cierre
last_days = dataset['Close'][-SEQLEN:].values.reshape(-1,1)

# Normalizar
last_days_scaled = scaler.transform(last_days)

# Lista para guardar predicciones escaladas
predictions_scaled = []

# Secuencia de entrada inicial con la forma que espera el modelo (1, SEQLEN, 1)
input_seq = last_days_scaled.reshape(1, SEQLEN, 1)

for _ in range(forecast_days):
    pred_scaled = model.predict(input_seq)[0,0]
    predictions_scaled.append(pred_scaled)

    # Actualizar secuencia de entrada: quitar el primer valor, añadir la predicción al final
    input_seq = np.append(input_seq[:,1:,:], [[[pred_scaled]]], axis=1)

# Invertir escala para volver a precio original
predictions_30 = scaler.inverse_transform(np.array(predictions_scaled).reshape(-1,1))

# Crear rango de fechas para la predicción
future_dates = pd.bdate_range(start=last_date, periods=forecast_days + 1, freq='B')[1:]

# Graficar últimos 60 días + predicción 30 días
historical_dates = dataset.index[-SEQLEN:]
historical_prices = dataset['Close'][-SEQLEN:]

plt.figure(figsize=(14,7))
plt.plot(historical_dates, historical_prices, label='Últimos 60 días históricos')
plt.plot(future_dates, predictions_30.flatten(), marker='o', linestyle='--', color='red', label='Proyección 30 días')

plt.title('Precio de acciones Tesla: Últimos 60 días y proyección de 30 días')
plt.xlabel('Fecha')
plt.ylabel('Precio')
plt.legend()
plt.grid(True)

# Set y-axis limits
plt.ylim(0)

plt.show()

# Ahora realizamos entrenamiento del precio de las acciones usando GRU

In [ ]:
from keras.models import Sequential
from keras.layers import GRU, Dense, Input

# --- Modelo GRU ---
model_gru = Sequential([
    Input(shape=(X_train.shape[1], 1)),
    GRU(units=50, return_sequences=True),
    GRU(units=50),
    Dense(1)
])

model_gru.compile(optimizer='adam', loss='mean_squared_error')

history_gru = model_gru.fit(X_train, y_train, epochs=50, batch_size=32,validation_split=0.1)

In [ ]:
# Graficar el history de la pérdida de entrenamiento y validación
plt.figure(figsize=(10,6))
plt.plot(history_gru.history['loss'], label='Pérdida de Entrenamiento')
plt.plot(history_gru.history['val_loss'], label='Pérdida de Validación')
plt.title('Evolución de la Pérdida durante el Entrenamiento')
plt.xlabel('Épocas')
plt.ylabel('Pérdida')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Supongo que model_gru ya está entrenado

# Tomamos los últimos SEQLEN días de precios reales desde el dataset completo
last_days = dataset['Close'][-SEQLEN:].values  # Últimos SEQLEN precios reales

# Normalizamos esos últimos días con el mismo scaler que usaste en entrenamiento
last_days_scaled = scaler.transform(last_days.reshape(-1,1))

predictions_scaled = []

# Preparamos la secuencia de entrada con la forma correcta para el modelo
input_seq = last_days_scaled.reshape(1, SEQLEN, 1)  # forma (1, SEQLEN, 1)

for _ in range(10):  # predecir 10 días futuros
    pred = model_gru.predict(input_seq)[0,0]
    predictions_scaled.append(pred)

    # Actualizamos input_seq para la siguiente predicción:
    # Quitamos el primer valor y añadimos la predicción al final
    input_seq = np.append(input_seq[:, 1:, :], [[[pred]]], axis=1)

# Convertimos las predicciones a escala original para interpretarlas
predictions = scaler.inverse_transform(np.array(predictions_scaled).reshape(-1,1))

print("Predicciones para los próximos 10 días:")
print(predictions.flatten())


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Parámetros
SEQLEN = 60  # longitud de la secuencia que usaste para entrenar el modelo
forecast_days = 30

# Última fecha del dataset
last_date = dataset.index[-1]

# Últimos SEQLEN precios de cierre
last_days = dataset['Close'][-SEQLEN:].values.reshape(-1,1)

# Normalizar
last_days_scaled = scaler.transform(last_days)

# Lista para guardar predicciones escaladas
predictions_scaled = []

# Secuencia de entrada inicial con la forma que espera el modelo (1, SEQLEN, 1)
input_seq = last_days_scaled.reshape(1, SEQLEN, 1)

for _ in range(forecast_days):
    pred_scaled = model_gru.predict(input_seq)[0,0]
    predictions_scaled.append(pred_scaled)

    # Actualizar secuencia de entrada: quitar el primer valor, añadir la predicción al final
    input_seq = np.append(input_seq[:,1:,:], [[[pred_scaled]]], axis=1)

# Invertir escala para volver a precio original
predictions_30 = scaler.inverse_transform(np.array(predictions_scaled).reshape(-1,1))

# Crear rango de fechas para la predicción
future_dates = pd.bdate_range(start=last_date, periods=forecast_days + 1, freq='B')[1:]

# Graficar últimos 60 días + predicción 30 días
historical_dates = dataset.index[-SEQLEN:]
historical_prices = dataset['Close'][-SEQLEN:]

plt.figure(figsize=(14,7))
plt.plot(historical_dates, historical_prices, label='Últimos 60 días históricos')
plt.plot(future_dates, predictions_30.flatten(), marker='o', linestyle='--', color='red', label='Proyección 30 días')

plt.title('Precio de acciones Tesla: Últimos 60 días y proyección de 30 días')
plt.xlabel('Fecha')
plt.ylabel('Precio')
plt.legend()
plt.grid(True)
# Set y-axis limits
max_price = max(historical_prices.max(), predictions_30.max())
plt.ylim(0, max_price * 1.1) # Add a little padding to the top
plt.show()


In [ ]:
# Predicciones
predicted_prices = model_gru.predict(X_test)
predicted_prices = scaler.inverse_transform(predicted_prices)
# Calcular RMSE
rmse = math.sqrt(mean_squared_error(test_set, predicted_prices))
print(f'RMSE: {rmse}')

Entonces, GRU funciona mejor que LSTM en este caso. LSTM bidireccional también es una buena opción para hacer que el modelo sea más robusto. Pero esto puede variar según el conjunto de datos. Aplicar tanto LSTM como GRU juntos dio resultados aún mejores.


#Recomedaciones
## ¿Cómo se decide cuántas capas LSTM y cuántas neuronas usar?

Decidir cuántas capas LSTM y cuántas neuronas debe tener tu red LSTM depende de varios factores relacionados con el problema específico que estás resolviendo. No existe una única respuesta correcta, pero aquí te doy algunas pautas generales para tomar decisiones informadas.

### 1. **Número de capas LSTM**

El número de capas LSTM se elige según la **complejidad del problema**. Agregar más capas LSTM permite a la red aprender representaciones más abstractas y complejas de los datos, pero también incrementa el riesgo de **sobreajuste** (overfitting) y puede hacer que el modelo sea más difícil de entrenar.

#### Pautas comunes:
- **Redes más profundas** (más capas LSTM) son útiles cuando:
  - El problema es **muy complejo**, como en tareas de procesamiento de lenguaje natural (PLN), traducción de idiomas, etc.
  - Los datos tienen patrones temporales complejos que requieren múltiples capas para modelar adecuadamente.
  
- **Redes más simples** (menos capas LSTM) son más efectivas cuando:
  - El problema es **relativamente sencillo**, como tareas de predicción de series temporales.
  - Tienes pocos datos, ya que una red más profunda puede llevar a sobreajuste con pocos datos.

#### En la práctica:
- Comienza con **1 o 2 capas LSTM**. Si el modelo no está aprendiendo bien, puedes agregar más capas.
- En la mayoría de los casos, **2 a 4 capas LSTM** es una configuración común.

### 2. **Número de neuronas en cada capa LSTM**

El número de neuronas en cada capa LSTM se refiere a cuántas **unidades de memoria** tiene la red para aprender y representar secuencias de datos. Este valor también es un parámetro ajustable.

#### Factores a considerar:
- **Mayor número de neuronas**: permite que la red tenga más capacidad para aprender representaciones complejas de los datos, pero también aumenta el número de parámetros y el riesgo de sobreajuste si no se tiene suficiente cantidad de datos.
- **Menor número de neuronas**: hace que el modelo sea más simple y menos propenso a sobreajuste, pero puede no tener suficiente capacidad para aprender relaciones complejas.

#### Pautas comunes:
- Comienza con un número moderado de neuronas, como **50 o 100**.
- Si el modelo es demasiado simple y no aprende bien, intenta aumentar el número de neuronas.
- Si estás trabajando con secuencias más largas o complejas, puedes considerar usar más neuronas (por ejemplo, **200 o 300**), pero ten en cuenta que esto incrementa el tiempo de entrenamiento y el riesgo de sobreajuste.
- Si tienes pocos datos, es mejor comenzar con **menos neuronas** (por ejemplo, 30 a 50).

### 3. **Ajustando el número de capas y neuronas con validación cruzada**

La mejor manera de decidir el número de capas y neuronas es mediante **experimentación**. Generalmente, se realiza validación cruzada para evaluar cómo cambia el rendimiento del modelo a medida que ajustas estos hiperparámetros.

#### Pasos típicos:
- **Dividir los datos** en entrenamiento y validación.
- **Entrenar el modelo** con diferentes configuraciones (por ejemplo, 1, 2 o 3 capas LSTM con 50, 100, o 200 neuronas).
- **Evaluar el modelo** en el conjunto de validación.
- Elegir la configuración que **mejor generalice** en el conjunto de validación.

### 4. **Consideraciones adicionales**

Otros factores también pueden influir en la elección del número de capas y neuronas:

- **Cantidad de datos**: Si tienes un conjunto de datos pequeño, probablemente necesitarás un modelo más simple (menos capas y menos neuronas). Si tienes grandes volúmenes de datos, puedes permitirte redes más grandes.
- **Regularización**: Si estás usando **dropout**, **normalización** o **early stopping**, puedes permitirte usar más capas y neuronas sin sobreajustar tanto.
- **Tiempo de entrenamiento**: Modelos más grandes con más capas y neuronas son más lentos de entrenar, lo que puede ser un factor limitante.

### 5. **Ajuste de hiperparámetros con técnicas avanzadas**

Para optimizar aún más el modelo, puedes usar técnicas de **búsqueda de hiperparámetros** como:

- **Búsqueda aleatoria** (`Random Search`).
- **Búsqueda en cuadrícula** (`Grid Search`).
- **Optimización bayesiana** (con herramientas como **Optuna** o **Keras Tuner**).

### Resumen de consideraciones:

- **Número de capas LSTM**:
  - Comienza con 1 o 2 capas.
  - Más capas son útiles para problemas más complejos, pero pueden llevar a sobreajuste si no tienes suficientes datos.
- **Número de neuronas**:
  - Comienza con 50 o 100 neuronas por capa.
  - Aumenta las neuronas si el modelo necesita mayor capacidad, pero vigila el sobreajuste.
- **Experimentación**: Siempre valida tu modelo con un conjunto de datos de validación y ajusta según los resultados. Puedes probar diferentes configuraciones utilizando técnicas de búsqueda de hiperparámetros.

### Ejemplo de experimentación con Keras:

En Keras, puedes experimentar con diferentes configuraciones de capas y neuronas fácilmente:

```python
# Ejemplo con 1 capa LSTM
model_1 = Sequential()
model_1.add(LSTM(units=50, return_sequences=False, input_shape=(X_train.shape[1], 1)))
model_1.add(Dense(units=1))

# Ejemplo con 2 capas LSTM
model_2 = Sequential()
model_2.add(LSTM(units=50, return_sequences=True, input_shape=(X_train.shape[1], 1)))
model_2.add(LSTM(units=50, return_sequences=False))
model_2.add(Dense(units=1))

# Compilar y entrenar ambos modelos
model_1.compile(optimizer='adam', loss='mean_squared_error')
model_2.compile(optimizer='adam', loss='mean_squared_error')

# Entrenamiento (en cada modelo)
model_1.fit(X_train, y_train, epochs=10, batch_size=32)
model_2.fit(X_train, y_train, epochs=10, batch_size=32)




```python

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.wrappers.scikit_learn import KerasRegressor
from sklearn.model_selection import GridSearchCV

# Función para crear el modelo - sin valores por defecto en los hiperparámetros a optimizar
def create_model(units, dropout_rate):
    model = Sequential()
    model.add(LSTM(units=units, return_sequences=True, input_shape=(X_train.shape[1], 1)))
    model.add(Dropout(dropout_rate))
    model.add(LSTM(units=units, return_sequences=True))
    model.add(Dropout(dropout_rate))
    model.add(LSTM(units=units))
    model.add(Dropout(dropout_rate))
    model.add(Dense(units=1))
    model.compile(optimizer='rmsprop', loss='mean_squared_error')
    return model

# Envolver el modelo
model = KerasRegressor(build_fn=create_model, verbose=0)

# Definir el espacio de búsqueda de hiperparámetros
param_grid = {
    'units': [50, 100],
    'dropout_rate': [0.2, 0.3],
    'batch_size': [32],  # Puedes expandir si quieres más combinaciones
    'epochs': [50]
}

# Búsqueda de hiperparámetros
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=3, n_jobs=-1)

# Entrenar búsqueda
grid_search.fit(X_train, y_train)

# Resultados
print("Mejores hiperparámetros encontrados:", grid_search.best_params_)
print("Mejor puntuación (media de validación):", grid_search.best_score_)

# Evaluación del mejor modelo en X_test
best_model = grid_search.best_estimator_
test_loss = best_model.score(X_test, y_test)
print("Pérdida en el conjunto de prueba:", test_loss)

```



# Enlaces y Videos explicativos de RNN, LLSTM y GRUP

Explicación breve de LSTM y GRU
[Enlace](https://github.com/adiacla/bigdata/raw/refs/heads/master/LSTM%20y%20GRU.docx)



In [ ]:
%%html
<iframe width="560" height="315" src="https://www.youtube.com/embed/x6E44DDWg5Q?si=CEkqthhbFq5SB1qG" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>

In [ ]:
%%html
<iframe width="560" height="315" src="https://www.youtube.com/embed/LagcbjDkqJE?si=1SOdcNZrUNAyxl_1" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>


In [ ]:
%%html
<iframe width="560" height="315" src="https://www.youtube.com/embed/f6PaCo-NfJA?si=RbBWUU3IhHH6B9RD" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>

---

# ⚡ 9. GRU — Gated Recurrent Unit: La Alternativa Eficiente al LSTM

---

## ¿Qué es el GRU?

El **GRU** (Gated Recurrent Unit), propuesto por Cho et al. (2014), es una simplificación del LSTM que mantiene las ventajas de las puertas pero con **menos parámetros** y **entrenamiento más rápido**.

La idea central: en lugar de tres puertas (forget, input, output) y dos estados ($h_t$ y $C_t$), el GRU usa **dos puertas** y **un solo estado oculto**.

---

## Ecuaciones del GRU

### 1. Puerta de actualización (Update Gate) — $z_t$

Decide **cuánto del estado anterior conservar** y cuánto del nuevo candidato usar.

$$z_t = \sigma(W_z x_t + U_z h_{t-1} + b_z)$$

- Si $z_t \approx 1$: se retiene casi todo el estado anterior (memoria a largo plazo).
- Si $z_t \approx 0$: se actualiza completamente con información nueva.

### 2. Puerta de reset (Reset Gate) — $r_t$

Decide **cuánto del estado anterior "olvidar"** al calcular el nuevo candidato.

$$r_t = \sigma(W_r x_t + U_r h_{t-1} + b_r)$$

- Si $r_t \approx 0$: ignora el estado anterior → permite "reiniciar" la memoria.

### 3. Estado candidato — $\tilde{h}_t$

Información nueva propuesta para el estado oculto, modulada por la puerta de reset:

$$\tilde{h}_t = \tanh(W_h x_t + U_h (r_t \odot h_{t-1}) + b_h)$$

### 4. Estado oculto final — $h_t$

Interpolación entre el estado anterior y el candidato, controlada por la puerta de actualización:

$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$$

---

## Comparación GRU vs LSTM

| Aspecto | SimpleRNN | LSTM | GRU |
|---------|:---------:|:----:|:---:|
| Puertas | 0 | 3 (forget, input, output) | 2 (update, reset) |
| Estados | $h_t$ | $h_t + C_t$ | $h_t$ |
| Parámetros | Menos | Más | Intermedios |
| Gradiente | ❌ Desvanece | ✅ Estable | ✅ Estable |
| Velocidad | ✅ Rápido | Lento | ✅ Rápido |
| Secuencias largas | ❌ | ✅ | ✅ |
| Cuándo usar | Prototipado | Tareas complejas | Balance rendimiento/velocidad |

---

## El GRU como caso límite del LSTM

Si en el LSTM se fusionan la puerta de olvido y la puerta de entrada como complementarias, y se elimina el estado de celda $C_t$, se obtiene esencialmente el GRU. Por eso en muchas tareas su rendimiento es comparable.


In [ ]:
# ─────────────────────────────────────────────────────
# VISUALIZACIÓN DE LAS PUERTAS DEL GRU
# ─────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(x): return 1 / (1 + np.exp(-x))
def tanh(x): return np.tanh(x)

# Simular el paso de tiempo de una celda GRU (valores escalarizados para visualización)
np.random.seed(42)
T = 30  # pasos de tiempo

# Simular señales de las puertas y estado oculto para demostración
z = sigmoid(np.random.randn(T) * 0.5 + 0.2)   # update gate (tendencia a 0.5)
r = sigmoid(np.random.randn(T) * 0.5 - 0.1)   # reset gate
h = np.zeros(T)
h[0] = 0.1
for t in range(1, T):
    h_cand = tanh(np.random.randn() * 0.3 + r[t] * h[t-1] * 0.2)
    h[t] = (1 - z[t]) * h[t-1] + z[t] * h_cand

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(z, color='steelblue', linewidth=2, label='Update Gate $z_t$')
axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[0].fill_between(range(T), 0, z, alpha=0.15, color='steelblue')
axes[0].set_ylabel('Activación', fontsize=11)
axes[0].set_title('Puerta de Actualización (Update Gate) — controla cuánto se retiene vs actualiza', fontsize=11)
axes[0].legend(fontsize=10); axes[0].set_ylim(0, 1); axes[0].grid(True, alpha=0.3)

axes[1].plot(r, color='tomato', linewidth=2, label='Reset Gate $r_t$')
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[1].fill_between(range(T), 0, r, alpha=0.15, color='tomato')
axes[1].set_ylabel('Activación', fontsize=11)
axes[1].set_title('Puerta de Reset (Reset Gate) — controla cuánto del pasado "olvidar"', fontsize=11)
axes[1].legend(fontsize=10); axes[1].set_ylim(0, 1); axes[1].grid(True, alpha=0.3)

axes[2].plot(h, color='seagreen', linewidth=2, label='Estado oculto $h_t$')
axes[2].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[2].set_ylabel('Valor', fontsize=11)
axes[2].set_xlabel('Paso de tiempo', fontsize=11)
axes[2].set_title('Estado oculto $h_t$ — memoria que fluye a través del tiempo', fontsize=11)
axes[2].legend(fontsize=10); axes[2].grid(True, alpha=0.3)

plt.suptitle('Visualización de las puertas internas de una celda GRU', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
print("\nInterpretación:")
print(f"  Update gate promedio: {z.mean():.3f} → {'Tiende a conservar historia' if z.mean() > 0.5 else 'Tiende a actualizar con info nueva'}")
print(f"  Reset gate promedio:  {r.mean():.3f} → {'Poco olvido del pasado' if r.mean() > 0.5 else 'Activo olvido del pasado'}")


---

# 🔬 Visualización Comparativa de las Puertas del LSTM

A diferencia del GRU, el LSTM mantiene **dos flujos de información** separados: el **estado de celda** $C_t$ (memoria a largo plazo) y el **estado oculto** $h_t$ (memoria de trabajo).

## Las cuatro operaciones del LSTM en detalle

### 1. Puerta de olvido (Forget Gate): ¿Qué se borra de la memoria?
$$f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$$

### 2. Puerta de entrada (Input Gate): ¿Qué información nueva entra?
$$i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)$$
$$\tilde{C}_t = \tanh(W_C [h_{t-1}, x_t] + b_C)$$

### 3. Actualización del estado de celda: Combinar olvido + nueva info
$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$

### 4. Puerta de salida (Output Gate): ¿Qué se expone como estado oculto?
$$o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)$$
$$h_t = o_t \odot \tanh(C_t)$$

El símbolo $\odot$ denota **producto elemento a elemento** (Hadamard).


In [ ]:
# ─────────────────────────────────────────────────────
# VISUALIZACIÓN DE LAS PUERTAS DEL LSTM
# ─────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(x): return 1 / (1 + np.exp(-x))

np.random.seed(7)
T = 30

# Simular activaciones de puertas y estados para visualización pedagógica
f = sigmoid(np.random.randn(T) * 0.6 + 0.3)   # forget gate
i = sigmoid(np.random.randn(T) * 0.6 - 0.2)   # input gate
o = sigmoid(np.random.randn(T) * 0.5 + 0.1)   # output gate
C_tilde = np.tanh(np.random.randn(T) * 0.5)    # cell candidate

C = np.zeros(T)
h = np.zeros(T)
for t in range(1, T):
    C[t] = f[t] * C[t-1] + i[t] * C_tilde[t]
    h[t] = o[t] * np.tanh(C[t])

fig, axes = plt.subplots(5, 1, figsize=(13, 12), sharex=True)
configs = [
    (f, 'tomato',    'Forget Gate $f_t$ — ¿Qué se borra de la memoria?', (0, 1)),
    (i, 'steelblue', 'Input Gate $i_t$ — ¿Cuánta info nueva entra?', (0, 1)),
    (o, 'purple',    'Output Gate $o_t$ — ¿Qué se expone como salida?', (0, 1)),
    (C, 'orange',    'Estado de Celda $C_t$ — Memoria a largo plazo', None),
    (h, 'seagreen',  'Estado Oculto $h_t$ — Memoria de trabajo / salida', None),
]

for ax, (signal, color, title, ylim) in zip(axes, configs):
    ax.plot(signal, color=color, linewidth=2)
    ax.fill_between(range(T), 0, signal, alpha=0.12, color=color)
    if ylim: ax.set_ylim(ylim); ax.axhline(0.5, color='gray', linestyle='--', alpha=0.4)
    ax.axhline(0, color='gray', linestyle='--', alpha=0.3)
    ax.set_title(title, fontsize=10, pad=3)
    ax.set_ylabel('Valor', fontsize=9)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Paso de tiempo', fontsize=11)
plt.suptitle('Dinámica interna de una celda LSTM a través del tiempo', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print("\nNota pedagógica:")
print("  • El estado de celda C_t puede mantener información durante muchos pasos (memoria larga)")
print("  • El estado oculto h_t es más 'volátil' y representa la memoria de trabajo")
print("  • Las puertas close to 0 o 1 actúan como interruptores on/off")


---

# 📐 2. Matemática de las RNNs — Forward y Backward Pass (Ampliado)

---

## BPTT — Backpropagation Through Time

Cuando desenrollamos una RNN a través de $T$ pasos de tiempo, obtenemos un grafo computacional profundo. El algoritmo **BPTT** aplica la regla de la cadena hacia atrás a través de este grafo.

### El problema del gradiente

El gradiente de la pérdida con respecto al estado oculto en el paso $t$ depende de los gradientes de **todos los pasos futuros**:

$$\frac{\partial L}{\partial h_t} = \frac{\partial L}{\partial h_T} \cdot \prod_{k=t}^{T-1} \frac{\partial h_{k+1}}{\partial h_k}$$

Cada factor $\frac{\partial h_{k+1}}{\partial h_k} = W^T \cdot \text{diag}(f'(h_k))$.

Si los valores propios de $W$ son $< 1$ → gradientes **desaparecen** exponencialmente.  
Si los valores propios de $W$ son $> 1$ → gradientes **explotan** exponencialmente.

### Soluciones implementadas en LSTM/GRU

| Problema | Solución en LSTM/GRU |
|----------|---------------------|
| Gradiente desvanecido | Conexiones de identidad en $C_t$ (LSTM) y $h_t$ (GRU) |
| Gradiente explosivo | Gradient Clipping (`clipnorm`, `clipvalue`) |
| Inestabilidad numérica | Funciones de activación $\sigma$ y $\tanh$ saturadas solo en extremos |

---

## Gradient Clipping en Keras

Keras aplica gradient clipping directamente en el optimizador:

```python
optimizer = tf.keras.optimizers.Adam(clipnorm=1.0)   # Norma máxima del gradiente
# o
optimizer = tf.keras.optimizers.Adam(clipvalue=0.5)  # Valor máximo por componente
```


In [ ]:
# ─────────────────────────────────────────────────────
# DEMOSTRACIÓN DEL PROBLEMA DEL GRADIENTE DESVANECIDO
# ─────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

def sigmoid_deriv(x):
    s = 1 / (1 + np.exp(-x))
    return s * (1 - s)

def tanh_deriv(x):
    return 1 - np.tanh(x)**2

# Simular propagación del gradiente hacia atrás en T pasos
T = 50
grad_simple_rnn = np.zeros(T)
grad_lstm_approx = np.zeros(T)

W_rnn  = 0.9   # Peso recurrente en SimpleRNN (< 1 → desvanece)
W_lstm = 1.0   # En LSTM el camino por C_t tiene ganancia ≈ 1

grad_simple_rnn[T-1] = 1.0
grad_lstm_approx[T-1] = 1.0

for t in range(T-2, -1, -1):
    # SimpleRNN: gradiente se multiplica por W * tanh'(activación)
    deriv = tanh_deriv(np.random.randn() * 0.5)   # ≈ 0.4–1.0
    grad_simple_rnn[t] = grad_simple_rnn[t+1] * W_rnn * deriv

    # LSTM: el camino por C_t tiene ganancia ≈ f_t ≈ 0.7–0.95 (puerta de olvido)
    forget = np.random.uniform(0.75, 0.95)
    grad_lstm_approx[t] = grad_lstm_approx[t+1] * forget

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(range(T-1, -1, -1), grad_simple_rnn, color='tomato', linewidth=2,
         label='SimpleRNN')
ax1.plot(range(T-1, -1, -1), grad_lstm_approx, color='steelblue', linewidth=2,
         label='LSTM (camino por $C_t$)')
ax1.set_title('Magnitud del gradiente a través del tiempo', fontsize=12)
ax1.set_xlabel('Pasos hacia atrás en el tiempo')
ax1.set_ylabel('Magnitud del gradiente')
ax1.legend(); ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

ax2.semilogy(range(T-1, -1, -1), grad_simple_rnn + 1e-15, color='tomato',
             linewidth=2, label='SimpleRNN (log scale)')
ax2.semilogy(range(T-1, -1, -1), grad_lstm_approx, color='steelblue',
             linewidth=2, label='LSTM (log scale)')
ax2.set_title('Escala logarítmica — diferencia exponencial', fontsize=12)
ax2.set_xlabel('Pasos hacia atrás en el tiempo')
ax2.set_ylabel('log(Magnitud del gradiente)')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('Problema del Gradiente Desvanecido: SimpleRNN vs LSTM', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print(f"\nGradiente en paso 0 (SimpleRNN): {grad_simple_rnn[0]:.2e}")
print(f"Gradiente en paso 0 (LSTM):      {grad_lstm_approx[0]:.4f}")
print("\nEl LSTM mantiene gradientes útiles; el SimpleRNN los pierde casi completamente.")


---

# 🏗️ 3. Tipos de Arquitecturas RNN — Guía Visual

---

Dependiendo de la **longitud de entrada y salida**, las RNNs se configuran de diferente manera.

## One-to-One
- No es realmente recurrente: es una red densa clásica.
- Entrada única → Salida única.
- **Ejemplo**: Clasificación de imagen (aunque CNN es más apropiado).

## One-to-Many
- Una sola entrada → Secuencia de salidas.
- **Ejemplos**: Generación de texto desde una imagen (image captioning), generación de música desde un género.

```python
# En Keras: return_sequences=True en la capa RNN, Dense aplicada en cada paso
LSTM(units=128, return_sequences=True)
TimeDistributed(Dense(vocab_size, activation='softmax'))
```

## Many-to-One
- Secuencia de entradas → Una sola salida.
- **Ejemplos**: Análisis de sentimientos, clasificación de texto, predicción de precio.

```python
# En Keras: return_sequences=False (default)
LSTM(units=128)   # solo retorna el estado final
Dense(1, activation='sigmoid')
```

## Many-to-Many (sincronizado)
- Secuencia de entradas → Secuencia de salidas (misma longitud).
- **Ejemplos**: Reconocimiento de entidades (NER), etiquetado POS.

```python
LSTM(units=128, return_sequences=True)
TimeDistributed(Dense(num_tags, activation='softmax'))
```

## Many-to-Many (no sincronizado / seq2seq)
- Secuencia de entradas → Secuencia de salidas (diferente longitud).
- **Ejemplos**: Traducción automática, resumen de texto, chatbots.
- Requiere arquitectura **Encoder-Decoder**.

```python
# Encoder
encoder_out, state_h, state_c = LSTM(256, return_state=True)(encoder_input)
# Decoder
decoder_out = LSTM(256)(decoder_input, initial_state=[state_h, state_c])
```


---

# ⚗️ Experimento: Comparación SimpleRNN vs LSTM vs GRU en Series Temporales

En esta sección comparamos las tres arquitecturas en una misma tarea de predicción de serie temporal para ver empíricamente las diferencias en rendimiento y velocidad de convergencia.


In [ ]:
# ─────────────────────────────────────────────────────
# COMPARACIÓN EMPÍRICA: SimpleRNN vs LSTM vs GRU
# en predicción de series temporales
# ─────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
import time

tf.random.set_seed(42)
np.random.seed(42)

# ── Generar serie temporal con dependencias de largo plazo ──
n = 500
t = np.linspace(0, 10 * np.pi, n)
# Serie con dos frecuencias (requiere capturar dependencias medianas-largas)
serie = np.sin(t) + 0.5 * np.sin(3 * t) + 0.1 * np.random.randn(n)

# Normalizar
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
serie_scaled = scaler.fit_transform(serie.reshape(-1, 1)).flatten()

# Crear secuencias
WINDOW = 20
X_all, y_all = [], []
for i in range(len(serie_scaled) - WINDOW):
    X_all.append(serie_scaled[i:i+WINDOW])
    y_all.append(serie_scaled[i+WINDOW])

X_all = np.array(X_all).reshape(-1, WINDOW, 1)
y_all = np.array(y_all)

split = int(len(X_all) * 0.8)
X_tr, X_ts = X_all[:split], X_all[split:]
y_tr, y_ts = y_all[:split], y_all[split:]

print(f"Train: {X_tr.shape} | Test: {X_ts.shape}")

# ── Definir modelos ──
def build_model(tipo, window, units=64):
    rnn_map = {
        'SimpleRNN': layers.SimpleRNN(units, dropout=0.1),
        'LSTM':      layers.LSTM(units, dropout=0.1),
        'GRU':       layers.GRU(units, dropout=0.1),
    }
    m = models.Sequential([
        layers.Input(shape=(window, 1)),
        rnn_map[tipo],
        layers.Dense(1)
    ], name=tipo)
    m.compile(optimizer='adam', loss='mse')
    return m

# ── Entrenar y medir ──
resultados = {}
for tipo in ['SimpleRNN', 'LSTM', 'GRU']:
    t0 = time.time()
    m = build_model(tipo, WINDOW)
    h = m.fit(X_tr, y_tr, epochs=40, batch_size=32,
              validation_data=(X_ts, y_ts), verbose=0)
    elapsed = time.time() - t0
    preds = scaler.inverse_transform(m.predict(X_ts, verbose=0))
    y_real = scaler.inverse_transform(y_ts.reshape(-1,1))
    rmse = np.sqrt(np.mean((preds - y_real)**2))
    resultados[tipo] = {'history': h.history, 'rmse': rmse, 'tiempo': elapsed, 'preds': preds}
    print(f"{tipo:12s} → RMSE: {rmse:.4f} | Tiempo: {elapsed:.1f}s | Params: {m.count_params():,}")

# ── Visualizar ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colores = {'SimpleRNN': 'tomato', 'LSTM': 'steelblue', 'GRU': 'seagreen'}

# Curvas de pérdida
for tipo, res in resultados.items():
    axes[0].plot(res['history']['val_loss'], label=tipo, color=colores[tipo])
axes[0].set_title('Val Loss por época')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('MSE')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# RMSE comparativo
tipos = list(resultados.keys())
rmses = [resultados[t]['rmse'] for t in tipos]
bars = axes[1].bar(tipos, rmses, color=[colores[t] for t in tipos], alpha=0.8, edgecolor='black')
axes[1].bar_label(bars, fmt='%.4f', fontsize=10)
axes[1].set_title('RMSE en conjunto de prueba')
axes[1].set_ylabel('RMSE'); axes[1].grid(True, alpha=0.3, axis='y')

# Predicciones vs real (últimos 60 puntos)
y_real_plot = scaler.inverse_transform(y_ts[-60:].reshape(-1,1)).flatten()
axes[2].plot(y_real_plot, color='black', linewidth=2, label='Real', alpha=0.7)
for tipo, res in resultados.items():
    axes[2].plot(res['preds'][-60:].flatten(), color=colores[tipo],
                 linestyle='--', linewidth=1.5, label=tipo, alpha=0.8)
axes[2].set_title('Predicciones vs Real (últimos 60 puntos)')
axes[2].set_xlabel('Tiempo'); axes[2].set_ylabel('Valor')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle('Comparación empírica: SimpleRNN vs LSTM vs GRU', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


---

# 📌 13. Resumen Comparativo y Cuándo Usar Cada Arquitectura

---

## Tabla resumen

| | SimpleRNN | LSTM | GRU |
|--|:---------:|:----:|:---:|
| **Año de introducción** | 1986 | 1997 | 2014 |
| **Parámetros** | Menos | Más | Intermedios |
| **Dependencias largas** | ❌ No | ✅ Sí | ✅ Sí |
| **Velocidad de entrenamiento** | ✅ Rápido | ❌ Lento | ✅ Rápido |
| **Cuándo usarlo** | Prototipado rápido, secuencias cortas | Texto complejo, audio, traducción | Balance velocidad/precisión |

## Cuándo usar cada variante de LSTM/GRU

| Variante | Cuándo | Código |
|---------|--------|--------|
| **LSTM/GRU simple** | Primera opción para la mayoría de tareas | `LSTM(units)` |
| **Apilado** | Datos muy complejos, vocabulario grande | `LSTM(units, return_sequences=True)` → `LSTM(units)` |
| **Bidireccional** | Clasificación de texto, NER (el contexto futuro importa) | `Bidirectional(LSTM(units))` |
| **Apilado + Bidireccional** | Traducción, modelos de comprensión lectora | Combinar ambos |

## El panorama actual

Las RNNs (LSTM/GRU) fueron el **estado del arte** en NLP de 2014 a 2017. Hoy, los **Transformers** (BERT, GPT) dominan NLP gracias a su capacidad de procesar secuencias en paralelo y capturar contexto global.

Sin embargo, las RNNs siguen siendo relevantes para:
- **Inferencia en tiempo real** (bajo costo computacional).
- **Dispositivos edge** con memoria limitada.
- **Series temporales** con ventanas cortas.
- **Aprendizaje pedagógico** de secuencias.

```
Cronología del estado del arte en NLP:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1986   SimpleRNN
1997   LSTM ──────────────────────────────┐
2014   GRU ─────────────────────────────┐ │
2015   Seq2Seq + Attention ─────────────┼─┘
2017   Transformer ─────────────────────┘
2018   BERT / GPT
2020+  GPT-3, T5, LLaMA, Gemini...
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
```


---

# 🏋️ 14. Ejercicios Propuestos

---

**Ejercicio 1 — Explorar window_size**
En el Ejemplo 1 (predicción de serie temporal), varía el `window_size` entre 5, 15 y 30. ¿Cómo afecta el RMSE final? ¿Por qué una ventana demasiado pequeña o demasiado grande puede ser problemática?

**Ejercicio 2 — Gradient Clipping**
Modifica el SimpleRNN del Ejemplo 1 para usar gradient clipping:
```python
model.compile(optimizer=tf.keras.optimizers.Adam(clipnorm=1.0), loss='mse')
```
¿Mejora la convergencia en secuencias más largas?

**Ejercicio 3 — LSTM apilado con Dropout**
Agrega una segunda capa LSTM al modelo de predicción de acciones Tesla:
```python
LSTM(100, return_sequences=True)
Dropout(0.3)
LSTM(50)
```
¿Mejora el RMSE? ¿Hay sobreajuste?

**Ejercicio 4 — GRU Bidireccional para clasificación**
Adapta el modelo de generación de nombres para usar un GRU Bidireccional. ¿Los nombres generados son más coherentes?

**Ejercicio 5 — Temperatura y creatividad**
En el Ejemplo 2 (generación de nombres), genera 20 nombres con temperatura = 0.3 y 20 con temperatura = 1.5. Clasifica subjetivamente cuántos son "reales" vs "inventados". ¿A qué temperatura el modelo es más creativo sin perder coherencia?

**Ejercicio 6 — Comparación con datos reales**
Descarga datos de otra acción (por ejemplo Apple: `AAPL` o Amazon: `AMZN`) y compara el RMSE de LSTM vs GRU. ¿Coincide con el resultado de Tesla?

**Ejercicio 7 — Búsqueda de hiperparámetros con Keras Tuner**
Implementa una búsqueda de hiperparámetros con Keras Tuner para el modelo de predicción de acciones, explorando:
- `units` en [32, 64, 128]
- `dropout` en [0.1, 0.2, 0.3]
- `learning_rate` en [1e-3, 1e-4]

**Ejercicio 8 — Conexión con el Módulo 11**
Después de completar este cuaderno, abra el **Cuaderno 11** (NLP con Embeddings). Identifique en ese cuaderno dónde se usa cada concepto aprendido aquí:
- ¿Dónde se usa la capa LSTM?
- ¿Cómo se conecta la capa Embedding con la capa LSTM?
- ¿Qué representa el `return_sequences` en ese contexto?
